# 04 — Estudo temporal da geração para estimar CO₂

Iatan Unit 2, ORISPL 6065, 2025. Esta etapa testa se o histórico da geração acrescenta informação à carga observada na própria hora para estimar a massa horária de CO₂. A matriz de defasagens usa somente geração elétrica atual e passada. O Parquet das partições é consumido pela etapa seguinte.

**Execução.** Em Colab, o padrão é a pasta de execução desta entrega. Em outro ambiente, execute na raiz do projeto ou defina IATAN_PROJECT_ROOT. A entrada padrão é data/interim/step05_feature_engineering.parquet, produzida na etapa de tratamento, e pode ser explicitada com IATAN_HOURLY_PARQUET. A primeira célula cria uma pasta única em outputs/lag_selection_predictive para toda a execução. Execute as células em ordem no mesmo kernel. Use pandas, numpy, pyarrow, scikit-learn, statsmodels, matplotlib e joblib.

**Comparação opcional de calendário.** IATAN_CALENDAR_REFERENCE indica um Parquet com timestamp. A comparação usa valores temporais tipados. Havendo fuso na referência ou na origem, informe IATAN_CEMS_TIMEZONE para interpretar as horas sem fuso se a convenção estiver documentada. Por exemplo, Etc/GMT+6 representa UTC−06:00 fixo. A convenção de fuso se aplica à conferência opcional. A análise mantém o eixo horário de origem.

**Estado da evidência.** As saídas do notebook original foram arquivadas e as desta cópia foram limpas. Os resultados do teste de 2025 já foram vistos pelo grupo. Uma execução revisada dessa avaliação será retrospectiva. A revisão de caminhos, referências e cobertura preserva os cortes, lags, modelos e hiperparâmetros do estudo.


In [ ]:
# Registros do experimento usam caminhos relativos à raiz configurada.
# Versões de bibliotecas e resultados são preservados sem identidade da sessão.
import builtins as _builtins
import json as _json
import os as _os
import re as _re
import warnings as _warnings
from pathlib import Path as _Path

def _project_root():
    for name in ("PROJECT_ROOT", "ROOT33", "ROOT", "PROJECT", "SOURCE", "root"):
        value = globals().get(name)
        if isinstance(value, _Path) and value.is_dir():
            return value.resolve()
    return _Path.cwd()

def portable_path(path, root=None):
    path = _Path(path).expanduser()
    root = _Path(root) if root is not None else _project_root()
    try:
        return path.resolve().relative_to(root.resolve()).as_posix()
    except (ValueError, OSError):
        return "external/" + path.name

def public_record(value):
    if isinstance(value, _Path):
        return portable_path(value)
    if isinstance(value, dict):
        return {k: public_record(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [public_record(v) for v in value]
    if isinstance(value, str):
        root = _project_root()
        for prefix in {str(root), root.as_posix()}:
            if value.startswith(prefix + "/") or value.startswith(prefix + "\\"):
                return value[len(prefix)+1:].replace("\\", "/")
            value = value.replace(prefix + "/", "").replace(prefix + "\\", "")
            value = value.replace(prefix, ".")
        if _re.match(r"^(?:[A-Za-z]:[\\/]|/(?:content|tmp|home|Users|mnt)/)", value):
            return "external/" + value.replace("\\", "/").rstrip("/").rsplit("/", 1)[-1]
        return value
    # Cópias de tabelas exportadas normalizam somente colunas textuais.
    if type(value).__module__.startswith("pandas.") and hasattr(value, "columns"):
        result = value.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(public_record)
        return result
    return value

def _project_print(*values, **kwargs):
    root = _project_root()
    if root.is_dir():
        _os.chdir(root)
    return _builtins.print(*(public_record(v) for v in values), **kwargs)

def _project_dump(value, *args, **kwargs):
    return _json.dump(public_record(value), *args, **kwargs)

def _project_dumps(value, *args, **kwargs):
    return _json.dumps(public_record(value), *args, **kwargs)

def _project_csv(frame, *args, **kwargs):
    return public_record(frame).to_csv(*args, **kwargs)

def _project_write_text(path, text, *args, **kwargs):
    return path.write_text(public_record(text), *args, **kwargs)

def _project_display(*values, **kwargs):
    from IPython.display import display as _display
    return _display(*(public_record(v) for v in values), **kwargs)

# Avisos mantêm o conteúdo sem diretórios temporários ou IDs do kernel.
_warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {public_record(str(message))}\n"
)
_os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
_context = _json.loads(_os.environ.get("MLFLOW_RUN_CONTEXT", "{}"))
_context.update({
    "mlflow.user": "equipe", "mlflow.source.name": "iatan",
    "mlflow.source.type": "LOCAL", "mlflow.source.git.commit": "",
    "mlflow.source.git.branch": "", "mlflow.source.git.repoURL": "",
    "mlflow.source.git.diff": "", "mlflow.source.git.dirty": "",
})
_os.environ["MLFLOW_RUN_CONTEXT"] = _json.dumps(_context)

# Referência metodológica: Aula 08, pp. 8–9, 31–32 e 53–56. Inventário, protocolo e identidade da execução.
# ============================================================
# SELEÇÃO PREDITIVA DE LAGS — CÉLULA 01/14
# CONFIGURAÇÃO, PROVENIÊNCIA E INVENTÁRIO
# ============================================================
# Esta célula NÃO abre os dados analíticos, NÃO constrói lags,
# NÃO treina modelos e NÃO lê features de processos.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import sys

try:
    from google.colab import drive
except ImportError:
    default_root = Path.cwd()
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    default_root = Path(
        '/content/drive/MyDrive/Ciência de Dados - Iatan - Process Mining e Experimentos'
        '/Iatan 2025 - entrega revisada/execucao'
    )

# Uma fonte explícita para toda a execução, sem procurar versões alternativas.
PROJECT_ROOT = Path(os.getenv('IATAN_PROJECT_ROOT', str(default_root))).expanduser().resolve()
hourly_input = Path(os.getenv('IATAN_HOURLY_PARQUET', 'data/interim/step05_feature_engineering.parquet')).expanduser()
HOURLY_SOURCE = (PROJECT_ROOT / hourly_input).resolve()
calendar_input = os.getenv('IATAN_CALENDAR_REFERENCE')
CALENDAR_REFERENCE = (PROJECT_ROOT / Path(calendar_input).expanduser()).resolve() if calendar_input else None
CEMS_TIMEZONE = os.getenv('IATAN_CEMS_TIMEZONE') or None
if CALENDAR_REFERENCE is not None and not CALENDAR_REFERENCE.is_file():
    raise FileNotFoundError(f'Referência de calendário declarada não encontrada: {CALENDAR_REFERENCE}')

# 2. PARÂMETROS DO PROTOCOLO — sujeitos à auditoria na Célula 02
UNIT_NAME = 'Iatan Unit 2'
ORISPL = 6065
UNIT_ID = '2'
YEAR = 2025
MAX_LAG_HOURS = 168
HORIZONS_HOURS = [0, 1, 3, 6, 12, 24, 36, 48, 72, 96, 120, 144, 168]
PROBE_MODELS = ['ridge', 'random_forest']
PRIMARY_METRIC = 'RMSE_CO2_metric_tons_per_hour_ON'
SEED = 42

# 3. VERIFICAR ARQUIVO OBRIGATÓRIO, SEM ABRIR O PARQUET
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Pasta do projeto ausente: {PROJECT_ROOT}')
if not HOURLY_SOURCE.is_file():
    raise FileNotFoundError(
        f'Série horária aprovada não encontrada: {HOURLY_SOURCE}. '
        'Confirme o caminho antes de prosseguir; não substitua por arquivo de processos.'
    )

# 4. CRIAR EXECUÇÃO ISOLADA — nunca sobrescrever a Fase 3 anterior
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = (PROJECT_ROOT / 'outputs/lag_selection_predictive'
           / f'iatan_6065_unit2_2025_{RUN_ID}')
REPORTS_DIR = RUN_DIR / 'reports'
DATA_DIR = RUN_DIR / 'data/processed'
FIGURES_DIR = RUN_DIR / 'figures'
for folder in (REPORTS_DIR, DATA_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=False)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

inputs = {
    'phase1_hourly_generation_co2': {
        'path': str(HOURLY_SOURCE),
        'size_bytes': HOURLY_SOURCE.stat().st_size,
        'sha256': sha256_file(HOURLY_SOURCE),
        'role': 'única fonte analítica obrigatória; esquema será auditado na Célula 02'
    }
}
if CALENDAR_REFERENCE is not None:
    inputs['optional_calendar_audit_reference'] = {
        'path': str(CALENDAR_REFERENCE),
        'size_bytes': CALENDAR_REFERENCE.stat().st_size,
        'sha256': sha256_file(CALENDAR_REFERENCE),
        'role': 'referência opcional de calendário; sem usar features de processos'
    }

configuration = {
    'study': 'predictive_generation_lag_selection',
    'cell': 1, 'total_cells': 14, 'run_id': RUN_ID,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'unit': {'name': UNIT_NAME, 'orispl': ORISPL, 'unit_id': UNIT_ID, 'year': YEAR},
    'objective': 'estimar CO2 contemporâneo a partir da geração atual e passada',
    'max_lag_hours': MAX_LAG_HOURS,
    'candidate_cumulative_horizons_hours': HORIZONS_HOURS,
    'probe_models': PROBE_MODELS,
    'primary_metric': PRIMARY_METRIC,
    'random_seed': SEED,
    'rules': [
        'somente geracao eletrica atual e historica como preditores',
        'CO2 observado exclusivamente como alvo',
        'sem HMM, Process Mining ou features de processos',
        'sem CO2 defasado e sem informacao futura',
        'definir e congelar protocolo temporal antes da selecao',
        'teste reservado ate o congelamento do conjunto final',
        'comparar horizontes sobre amostras equivalentes',
        'distinguir horas OFF, lacunas e inicio de ciclos',
        'nao alterar artefatos das fases anteriores'
    ],
    'paths': {'project_root': str(PROJECT_ROOT), 'run_dir': str(RUN_DIR)},
    'inputs': inputs,
    'timestamp_policy': {
        'analysis_axis': 'preserved_source_hour',
        'naive_timezone_for_optional_comparison': CEMS_TIMEZONE,
    },
    'environment': {'python': sys.version.split()[0]},
    'status': 'cell01_configured_schema_not_yet_audited'
}
CONFIG_FILE = REPORTS_DIR / 'cell01_configuration_inventory.json'
_project_write_text(CONFIG_FILE, _project_dumps(configuration, ensure_ascii=False, indent=2), encoding='utf-8')

_project_print('=' * 76)
_project_print('CÉLULA 01/14 CONCLUÍDA — SELEÇÃO PREDITIVA DE LAGS')
_project_print('=' * 76)
_project_print(f'Unidade: {UNIT_NAME} | ORISPL {ORISPL} | Ano {YEAR}')
_project_print(f'Fonte horária aprovada: {HOURLY_SOURCE}')
_project_print(f'Referência opcional de calendário: {CALENDAR_REFERENCE}')
_project_print(f'Horizonte máximo: {MAX_LAG_HOURS} h | horizontes candidatos: {HORIZONS_HOURS}')
_project_print(f'Modelos auxiliares planejados: {PROBE_MODELS}')
_project_print(f'Pasta desta execução: {RUN_DIR}')
_project_print(f'Inventário SHA-256: {CONFIG_FILE}')
_project_print('SEM treinamento, seleção de lags ou leitura de dados de processos.')
_project_print('PRÓXIMA CÉLULA: 02 — auditoria de esquema, geração, CO2 e calendário.')


In [ ]:
# Referência metodológica: EPA CAMD Data Guide, pp. 1–2. Unidades e escopo dos dados horários. Comparação tipada dos calendários é controle do projeto.
# ================================================================
# SELEÇÃO PREDITIVA DE LAGS — CÉLULA 02/14
# AUDITORIA DA SÉRIE HORÁRIA EPA: GERAÇÃO, CO2 E CALENDÁRIO
# ================================================================
# Não usa features operacionais, não cria lags e não treina modelos.
from pathlib import Path
import pandas as pd
import numpy as np
import json, hashlib
from datetime import datetime, timezone

if 'RUN_DIR' not in globals() or 'RUN_ID' not in globals():
    raise RuntimeError('Execute a Célula 01 desta fase para configurar a execução.')

CONFIG_FILE = RUN_DIR / 'reports/cell01_configuration_inventory.json'
assert CONFIG_FILE.is_file(), f'Inventário da Célula 01 ausente: {CONFIG_FILE}'
config = json.loads(CONFIG_FILE.read_text(encoding='utf-8'))
assert config['study'] == 'predictive_generation_lag_selection'
assert config['unit']['year'] == 2025
if config['run_id'] != RUN_ID or Path(config['paths']['run_dir']).resolve() != RUN_DIR.resolve():
    raise RuntimeError('O manifesto não corresponde à execução iniciada nesta sessão.')
SOURCE = Path(config['inputs']['phase1_hourly_generation_co2']['path'])
assert SOURCE.is_file(), f'Fonte ausente: {SOURCE}'

def sha256_file(path, chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''):
            h.update(chunk)
    return h.hexdigest()

assert sha256_file(SOURCE) == config['inputs']['phase1_hourly_generation_co2']['sha256'], (
    'A fonte foi alterada desde a Célula 01; interromper e investigar.')

raw = pd.read_parquet(SOURCE)
assert len(raw)>0, 'Fonte vazia.'
assert raw.columns.is_unique, 'Há nomes de colunas duplicados.'
_project_print('Esquema original (somente auditoria):')
_project_print(raw.dtypes.to_string())
_project_print('Linhas de origem:',len(raw),'| colunas de origem:',len(raw.columns))

# Mapeamento EXPLÍCITO confirmado no esquema real da Fase 1.
# Usar os valores originais; colunas tratadas e flags permanecem fora
# do conjunto analítico. Não há HMM, Process Mining ou lags nesta etapa.
mapped = {
    'timestamp': 'timestamp',
    'generation_mw': 'Gross Load (MW)',
    'co2_short_tons': 'CO2 Mass (short tons)',
}
for role, col in mapped.items():
    assert col in raw.columns, f'Coluna ausente ({role}): {col}'
assert len(set(mapped.values())) == 3, 'Colunas mapeadas não são distintas.'
assert pd.api.types.is_datetime64_any_dtype(raw[mapped['timestamp']]), 'Timestamp inválido.'
for role in ['generation_mw', 'co2_short_tons']:
    assert pd.api.types.is_numeric_dtype(raw[mapped[role]]), f'Coluna não numérica: {mapped[role]}'
_project_print('\nMapeamento confirmado:', mapped)

# CO2 da EPA vem em short tons. Converter explicitamente para
# toneladas métricas: 1 short ton = 0.90718474 metric tons.
SHORT_TON_TO_METRIC_TON = 0.90718474
df = raw[[mapped['timestamp'], mapped['generation_mw'], mapped['co2_short_tons']]].copy()
df.columns = ['timestamp', 'generation_mw', 'co2_short_tons']
df['co2_metric_tons'] = df['co2_short_tons'] * SHORT_TON_TO_METRIC_TON

# Não converter timezone silenciosamente; timestamps ingênuos são
# mantidos no relógio de origem. A convenção deverá ser registrada.
ts=pd.to_datetime(df['timestamp'],errors='coerce')
assert ts.notna().all(), f'Timestamps inválidos: {int(ts.isna().sum())}'
assert not ts.duplicated().any(), f'Timestamps duplicados: {int(ts.duplicated().sum())}'
assert ts.dt.minute.eq(0).all() and ts.dt.second.eq(0).all(), 'Timestamps não estão em hora cheia.'
df['timestamp']=ts
df=df.sort_values('timestamp').reset_index(drop=True)
for c in ['generation_mw','co2_metric_tons']:
    df[c]=pd.to_numeric(df[c],errors='coerce')

# Calendário completo do ano, no mesmo padrão temporal da origem.
# Se o timestamp tiver timezone, pd.date_range preserva esse timezone.
zone=df['timestamp'].dt.tz
start=pd.Timestamp('2025-01-01 00:00:00',tz=zone)
end=pd.Timestamp('2025-12-31 23:00:00',tz=zone)
calendar=pd.DataFrame({'timestamp':pd.date_range(start,end,freq='h')})
assert len(calendar)==8760, 'Calendário 2025 inesperado.'
assert df['timestamp'].between(start,end).all(), 'A fonte contém timestamps fora de 2025.'

# Um único registro por hora; merge não inventa valores OFF nem CO2.
calendar=calendar.merge(df,on='timestamp',how='left',indicator='source_presence',validate='one_to_one')
calendar['hour_present_in_source']=calendar['source_presence'].eq('both')
calendar=calendar.drop(columns=['source_presence'])
calendar['generation_zero']=calendar['generation_mw'].eq(0) & calendar['generation_mw'].notna()
calendar['generation_positive']=calendar['generation_mw'].gt(0) & calendar['generation_mw'].notna()
calendar['generation_missing']=calendar['generation_mw'].isna()
calendar['co2_missing']=calendar['co2_metric_tons'].isna()
calendar['co2_zero']=calendar['co2_metric_tons'].eq(0) & calendar['co2_metric_tons'].notna()
calendar['co2_positive']=calendar['co2_metric_tons'].gt(0) & calendar['co2_metric_tons'].notna()
calendar['negative_generation']=calendar['generation_mw'].lt(0)
calendar['negative_co2']=calendar['co2_metric_tons'].lt(0)
calendar['generation_positive_co2_zero']=calendar['generation_positive'] & calendar['co2_zero']
calendar['generation_zero_co2_positive']=calendar['generation_zero'] & calendar['co2_positive']

# Ciclos provisórios definidos APENAS pela geração positiva observada.
# A definição operacional definitiva (e horas OFF) é assunto da Célula 03.
on=calendar['generation_positive'].fillna(False)
starts=on & ~on.shift(fill_value=False)
calendar['provisional_on_cycle']=starts.cumsum().where(on).astype('Int64')
cycles=calendar.loc[on].groupby('provisional_on_cycle',observed=True).agg(
    first_hour=('timestamp','min'),last_hour=('timestamp','max'),hours=('timestamp','size'))

# Auditoria opcional de calendário, sem modificar timestamps analíticos.
def compare_calendar_timestamps(reference_values, calendar_values, naive_timezone):
    try:
        reference_idx = pd.DatetimeIndex(pd.to_datetime(reference_values, errors='raise'))
        calendar_idx = pd.DatetimeIndex(pd.to_datetime(calendar_values, errors='raise'))
    except (ValueError, TypeError) as exc:
        return {'status': 'invalid_timestamps', 'detail': str(exc)}

    result = {'timestamps_unique': bool(not reference_idx.hasnans and reference_idx.is_unique)}
    if not result['timestamps_unique']:
        return {**result, 'status': 'missing_or_duplicate_reference_timestamps'}

    mixed_awareness = (reference_idx.tz is None) != (calendar_idx.tz is None)
    if mixed_awareness and naive_timezone is None:
        return {**result, 'status': 'timezone_required', 'timestamp_set_matches_2025_calendar': None}

    if reference_idx.tz is not None or calendar_idx.tz is not None:
        try:
            if reference_idx.tz is None:
                reference_idx = reference_idx.tz_localize(naive_timezone, ambiguous='raise', nonexistent='raise')
            if calendar_idx.tz is None:
                calendar_idx = calendar_idx.tz_localize(naive_timezone, ambiguous='raise', nonexistent='raise')
            reference_idx = reference_idx.tz_convert('UTC')
            calendar_idx = calendar_idx.tz_convert('UTC')
        except Exception as exc:
            raise ValueError(f'Fuso declarado incompatível com os timestamps: {naive_timezone!r}') from exc

    matches = reference_idx.sort_values().tolist() == calendar_idx.sort_values().tolist()
    return {**result, 'status': 'compared', 'timestamp_set_matches_2025_calendar': matches}


ref_audit = {'available': False, 'status': 'not_requested'}
ref = config['inputs'].get('optional_calendar_audit_reference')
if ref:
    refpath = Path(ref['path'])
    if not refpath.is_file() or sha256_file(refpath) != ref['sha256']:
        raise RuntimeError('O calendário opcional mudou desde o inventário desta execução.')
    ref_df = pd.read_parquet(refpath)
    ref_audit = {'available': True, 'rows': len(ref_df), 'sha256_verified': True}
    if 'timestamp' not in ref_df.columns:
        ref_audit['status'] = 'missing_timestamp_column'
    else:
        ref_audit.update(compare_calendar_timestamps(
            ref_df['timestamp'], calendar['timestamp'],
            config['timestamp_policy']['naive_timezone_for_optional_comparison'],
        ))

counts={
    'source_rows':int(len(df)), 'calendar_hours':int(len(calendar)),
    'hours_missing_from_source':int((~calendar['hour_present_in_source']).sum()),
    'generation_positive_hours':int(calendar['generation_positive'].sum()),
    'generation_zero_hours':int(calendar['generation_zero'].sum()),
    'generation_missing_hours':int(calendar['generation_missing'].sum()),
    'co2_positive_hours':int(calendar['co2_positive'].sum()),
    'co2_zero_hours':int(calendar['co2_zero'].sum()),
    'co2_missing_hours':int(calendar['co2_missing'].sum()),
    'negative_generation_hours':int(calendar['negative_generation'].sum()),
    'negative_co2_hours':int(calendar['negative_co2'].sum()),
    'positive_generation_zero_co2_hours':int(calendar['generation_positive_co2_zero'].sum()),
    'zero_generation_positive_co2_hours':int(calendar['generation_zero_co2_positive'].sum()),
    'provisional_on_cycles':int(len(cycles)),
    'max_provisional_on_cycle_hours':int(cycles['hours'].max()) if len(cycles) else 0,
}
_project_print('\nAUDITORIA — contagens:')
for k,v in counts.items(): _project_print(f'  {k}: {v}')
_project_print('\nResumo das variáveis:')
_project_print(calendar[['generation_mw','co2_short_tons','co2_metric_tons']].describe().to_string())
_project_print('\nCiclos ON provisórios (primeiros 10):')
_project_print(cycles.head(10).to_string())
_project_print('\nReferência opcional de calendário:',ref_audit)

# Bloqueios para inconsistências que comprometem a interpretação.
assert counts['negative_generation_hours']==0, 'Geração negativa: revisar unidade/sinal.'
assert counts['negative_co2_hours']==0, 'CO2 negativo: revisar unidade/sinal.'
assert counts['generation_positive_hours']>0, 'Nenhuma hora com geração positiva.'
assert counts['co2_positive_hours']>0, 'Nenhuma hora com CO2 positivo.'
assert not (calendar['generation_positive'] & calendar['co2_missing']).any(), (
    'Há horas ON sem CO2 medido: revisar antes de avançar.')

# Exportar auditoria sem imputar, sem selecionar lags e sem processos.
reports=RUN_DIR/'reports'; processed=RUN_DIR/'data/processed'
reports.mkdir(parents=True,exist_ok=True);processed.mkdir(parents=True,exist_ok=True)
out_parquet=processed/'cell02_generation_co2_calendar_audited.parquet'
out_cycles=reports/'cell02_provisional_on_cycles.csv'
out_report=reports/'cell02_hourly_source_audit.json'
calendar.to_parquet(out_parquet,index=False)
_project_csv(cycles.reset_index(), out_cycles,index=False)
report={
    'study':config['study'],'cell':2,'run_id':RUN_ID,'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_sha256':config['inputs']['phase1_hourly_generation_co2']['sha256'],
    'source_columns_mapped':mapped,'source_timestamp_timezone':str(zone),
    'units':{'generation_mw':'MW','co2_short_tons':'short tons/hour','co2_metric_tons':'metric tons/hour'},
    'co2_conversion_factor_short_to_metric':SHORT_TON_TO_METRIC_TON,
    'note':'CO2 convertido explicitamente de short tons para toneladas métricas; geração e CO2 originais preservados.',
    'counts':counts,'reference_calendar_audit':ref_audit,
    'policy_status':'ON/OFF and evaluation eligibility NOT frozen; Cell 03',
    'output':str(out_parquet),'output_sha256':sha256_file(out_parquet),
    'process_features_used':False,'lags_created':False,'models_trained':False
}
_project_write_text(out_report, _project_dumps(report,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
_project_print('\n'+'='*76)
_project_print('CÉLULA 02/14 CONCLUÍDA — AUDITORIA DE GERAÇÃO E CO2')
_project_print('='*76)
_project_print('Calendário auditado:',out_parquet)
_project_print('Ciclos provisórios:',out_cycles)
_project_print('Relatório:',out_report)
_project_print('Nenhum lag criado; nenhum modelo treinado; nenhuma feature de processos utilizada.')
_project_print('PRÓXIMA CÉLULA: 03 — protocolo experimental, ON/OFF e partições.')

In [ ]:
# Referência metodológica: Cerqueira et al. (2020) e Cawley e Talbot (2010). Ordem temporal e separação de seleção e avaliação. Cortes são do projeto.

# ============================================================
# CÉLULA 03/14 — PROTOCOLO EXPERIMENTAL E PARTIÇÕES
# VERSÃO CORRIGIDA
# ============================================================
#
# PROJETO: Seleção preditiva de lags de geração para CO2
# UNIDADE: Iatan Unit 2 | ORISPL 6065 | Ano 2025
#
# OBJETIVOS:
# 1. Carregar o calendário auditado da Célula 02.
# 2. Identificar observações elegíveis para modelagem.
# 3. Preservar zeros e valores ausentes.
# 4. Definir partições cronológicas.
# 5. Registrar o protocolo experimental.
#
# Não cria lags nem treina modelos.
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. DIRETÓRIOS
# ------------------------------------------------------------



INPUT_PATH = (
    RUN_DIR
    / "data/processed"
    / "cell02_generation_co2_calendar_audited.parquet"
)

DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_PATH.exists(), (
    f"Arquivo da Célula 02 não encontrado: {INPUT_PATH}"
)

# ------------------------------------------------------------
# 2. CARREGAMENTO E VALIDAÇÃO
# ------------------------------------------------------------

input_audit = json.loads((REPORT_DIR / 'cell02_hourly_source_audit.json').read_text(encoding='utf-8'))
if input_audit['run_id'] != RUN_ID or input_audit['output_sha256'] != sha256_file(INPUT_PATH):
    raise RuntimeError('O calendário não corresponde à auditoria desta execução.')
df = pd.read_parquet(INPUT_PATH).copy()

REQUIRED_COLUMNS = [
    "timestamp",
    "generation_mw",
    "co2_metric_tons",
]

missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in df.columns
]

assert not missing_columns, (
    f"Colunas ausentes: {missing_columns}"
)

TIME_COL = "timestamp"
GEN_COL = "generation_mw"
CO2_COL = "co2_metric_tons"

df[TIME_COL] = pd.to_datetime(
    df[TIME_COL], errors="raise"
)

df[GEN_COL] = pd.to_numeric(
    df[GEN_COL], errors="coerce"
)

df[CO2_COL] = pd.to_numeric(
    df[CO2_COL], errors="coerce"
)

df = df.sort_values(TIME_COL).reset_index(drop=True)

assert len(df) == 8760
assert df[TIME_COL].notna().all()
assert df[TIME_COL].is_unique

expected_calendar = pd.date_range(
    start="2025-01-01 00:00:00",
    periods=8760,
    freq="h",
    tz=df[TIME_COL].dt.tz,
)

assert np.array_equal(
    df[TIME_COL].to_numpy(),
    expected_calendar.to_numpy()
), "Calendário principal inconsistente."

_project_print("Fonte:", INPUT_PATH)
_project_print("Linhas:", len(df))
_project_print("Geração:", GEN_COL)
_project_print("CO2:", CO2_COL)

# ------------------------------------------------------------
# 3. ELEGIBILIDADE — CORRIGIDA
# ------------------------------------------------------------
#
# Usar observações com geração e CO2 registrados,
# ambos não negativos.
#
# Geração igual a zero é preservada.
# Valores ausentes NÃO são convertidos em zero.
# Não depende das colunas is_on ou Operating Time.
# ------------------------------------------------------------

df["eligible_for_modeling"] = (
    df[GEN_COL].notna()
    & df[CO2_COL].notna()
    & (df[GEN_COL] >= 0)
    & (df[CO2_COL] >= 0)
)

df["calendar_generation_missing"] = (
    df[GEN_COL].isna()
)

n_eligible = int(
    df["eligible_for_modeling"].sum()
)

n_not_eligible = int(
    (~df["eligible_for_modeling"]).sum()
)

n_zero_generation = int(
    (
        df["eligible_for_modeling"]
        & (df[GEN_COL] == 0)
    ).sum()
)

_project_print("\nELEGIBILIDADE")
_project_print("Horas elegíveis:", n_eligible)
_project_print("Horas não elegíveis:", n_not_eligible)
_project_print(
    "Horas elegíveis com geração zero:",
    n_zero_generation
)

assert n_eligible == 6249, (
    "Divergência em relação à auditoria da Célula 02."
)

# ------------------------------------------------------------
# 4. PARTIÇÕES CRONOLÓGICAS
# ------------------------------------------------------------
#
# Treino: 72%
# Validação: 11%
# Teste: aproximadamente 17%
#
# O teste será reservado até a Célula 13.
# ------------------------------------------------------------

eligible_idx = np.flatnonzero(
    df["eligible_for_modeling"].to_numpy()
)

n = len(eligible_idx)

n_train = int(np.floor(n * 0.72))
n_val = int(np.floor(n * 0.11))
n_test = n - n_train - n_val

assert n_train > 0
assert n_val > 0
assert n_test > 0

df["partition"] = "not_eligible"

df.loc[
    eligible_idx[:n_train],
    "partition"
] = "train"

df.loc[
    eligible_idx[n_train:n_train+n_val],
    "partition"
] = "validation"

df.loc[
    eligible_idx[n_train+n_val:],
    "partition"
] = "test"

# ------------------------------------------------------------
# 5. AUDITORIA DAS PARTIÇÕES
# ------------------------------------------------------------

summary = []

for partition in ["train", "validation", "test"]:

    subset = df[
        df["partition"] == partition
    ]

    summary.append({
        "partition": partition,
        "rows": int(len(subset)),
        "first_timestamp": str(
            subset[TIME_COL].min()
        ),
        "last_timestamp": str(
            subset[TIME_COL].max()
        ),
        "generation_zero_hours": int(
            (subset[GEN_COL] == 0).sum()
        ),
        "co2_zero_hours": int(
            (subset[CO2_COL] == 0).sum()
        ),
    })

summary_df = pd.DataFrame(summary)

assert summary_df["rows"].sum() == n

train_last = df.loc[
    df["partition"] == "train",
    TIME_COL
].max()

val_first = df.loc[
    df["partition"] == "validation",
    TIME_COL
].min()

val_last = df.loc[
    df["partition"] == "validation",
    TIME_COL
].max()

test_first = df.loc[
    df["partition"] == "test",
    TIME_COL
].min()

assert train_last < val_first
assert val_last < test_first

_project_print("\nPARTIÇÕES CRONOLÓGICAS")
_project_print(summary_df.to_string(index=False))

_project_print("\nTOTAL")
_project_print("Treino:", n_train)
_project_print("Validação:", n_val)
_project_print("Teste:", n_test)
_project_print("Não elegíveis:", n_not_eligible)

# ------------------------------------------------------------
# 6. REGISTRO DO PROTOCOLO
# ------------------------------------------------------------

protocol = {
    "cell": "03/14",
    "run_id": RUN_ID,
    "source_sha256": input_audit["source_sha256"],
    "timestamp_axis": "preserved_source_hour",
    "project": "predictive_lag_selection",
    "plant": "Iatan",
    "orispl": 6065,
    "unit": "2",
    "year": 2025,
    "source": str(INPUT_PATH),
    "timestamp_column": TIME_COL,
    "generation_column": GEN_COL,
    "target_column": CO2_COL,
    "target_unit": "metric_tons",
    "calendar_rows": int(len(df)),
    "eligible_rows": n_eligible,
    "non_eligible_rows": n_not_eligible,
    "split_method": (
        "chronological_eligible_observations"
    ),
    "split_proportions": {
        "train": 0.72,
        "validation": 0.11,
        "test": 0.17,
    },
    "partitions": summary,
    "max_lag_hours": 168,
    "lag_selection_models": [
        "ridge",
        "random_forest",
    ],
    "test_reserved_until_cell": 13,
    "missing_generation_imputed": False,
    "zero_generation_preserved": True,
    "co2_lags_allowed": False,
    "process_features_allowed": False,
}

OUTPUT_DATA = (
    DATA_DIR
    / "cell03_experimental_partitions.parquet"
)

OUTPUT_REPORT = (
    REPORT_DIR
    / "cell03_experimental_protocol.json"
)

OUTPUT_SUMMARY = (
    REPORT_DIR
    / "cell03_partition_summary.csv"
)

df.to_parquet(
    OUTPUT_DATA,
    index=False
)

_project_csv(summary_df, 
    OUTPUT_SUMMARY,
    index=False
)

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8"
) as f:
    protocol["partition_parquet_sha256"] = sha256_file(OUTPUT_DATA)
    _project_dump(
        protocol,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 7. VERIFICAÇÃO DOS ARQUIVOS
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

for path in [
    OUTPUT_DATA,
    OUTPUT_REPORT,
    OUTPUT_SUMMARY,
]:
    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 03/14 CONCLUÍDA — "
    "PROTOCOLO EXPERIMENTAL"
)
_project_print("=" * 72)

_project_print("Observações elegíveis:", n_eligible)
_project_print("Treino:", n_train)
_project_print("Validação:", n_val)
_project_print("Teste:", n_test)

_project_print(
    "Nenhum lag criado; "
    "nenhum modelo treinado."
)

_project_print(
    "PRÓXIMA CÉLULA: 04 — "
    "construção dos lags 1–168 h."
)


In [ ]:
# Referência metodológica: pandas.DataFrame.shift. Defasagem em calendário completo. Gu et al. (2023) contextualiza a previsão de emissões com CEMS.

# ============================================================
# CÉLULA 04/14 — CONSTRUÇÃO DOS LAGS DE GERAÇÃO (1–168 h)
# ============================================================
#
# PROJETO: Seleção preditiva de lags para estimação de CO2
# UNIDADE: Iatan Unit 2 | ORISPL 6065 | 2025
#
# OBJETIVOS:
# 1. Carregar o arquivo aprovado da Célula 03.
# 2. Construir os lags horários de geração de 1 a 168 h.
# 3. Preservar zeros e valores ausentes.
# 4. Auditar disponibilidade e alinhamento dos lags.
# 5. Salvar a matriz para as próximas células.
#
# Não treina modelos nem utiliza CO2 defasado.
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"

INPUT_PATH = (
    DATA_DIR / "cell03_experimental_partitions.parquet"
)

OUTPUT_DATA = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

OUTPUT_REPORT = (
    REPORT_DIR / "cell04_generation_lag_audit.json"
)

OUTPUT_AVAILABILITY = (
    REPORT_DIR / "cell04_lag_availability.csv"
)

MAX_LAG = 168

assert INPUT_PATH.exists(), (
    f"Arquivo da Célula 03 não encontrado: {INPUT_PATH}"
)

# ------------------------------------------------------------
# 2. CARREGAMENTO E VERIFICAÇÕES
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_PATH).copy()

required = [
    "timestamp",
    "generation_mw",
    "co2_metric_tons",
    "eligible_for_modeling",
    "partition",
]

missing = [c for c in required if c not in df.columns]

assert not missing, f"Colunas ausentes: {missing}"

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="raise"
)

df["generation_mw"] = pd.to_numeric(
    df["generation_mw"], errors="coerce"
)

df = df.sort_values("timestamp").reset_index(drop=True)

expected = pd.date_range(
    "2025-01-01 00:00:00",
    periods=8760,
    freq="h"
)

assert len(df) == 8760
assert df["timestamp"].is_unique
assert np.array_equal(
    df["timestamp"].to_numpy(),
    expected.to_numpy()
), "Calendário horário inconsistente."

assert int(df["eligible_for_modeling"].sum()) == 6249

partition_counts = (
    df["partition"].value_counts().to_dict()
)

assert partition_counts.get("train", 0) == 4499
assert partition_counts.get("validation", 0) == 687
assert partition_counts.get("test", 0) == 1063

original_generation = df["generation_mw"].copy()

_project_print("Fonte:", INPUT_PATH)
_project_print("Horas de calendário:", len(df))
_project_print("Horas elegíveis:", int(df["eligible_for_modeling"].sum()))
_project_print("Horizonte máximo:", MAX_LAG, "horas")

# ------------------------------------------------------------
# 3. CONSTRUÇÃO DOS LAGS
# ------------------------------------------------------------
#
# Cada lag utiliza exclusivamente geração passada.
#
# G(t-1), G(t-2), ..., G(t-168).
#
# Os deslocamentos são feitos no calendário completo,
# antes de qualquer filtragem de horas elegíveis.
#
# Assim, um lag representa horas reais de calendário,
# não a observação elegível anterior.
# ------------------------------------------------------------

lag_frame = pd.DataFrame({
    f"generation_lag_{lag:03d}h": original_generation.shift(lag)
    for lag in range(1, MAX_LAG + 1)
})
lag_columns = lag_frame.columns.tolist()
df = pd.concat([df, lag_frame], axis=1)

assert len(lag_columns) == MAX_LAG

# ------------------------------------------------------------
# 4. AUDITORIA DE ALINHAMENTO
# ------------------------------------------------------------

for lag in [1, 3, 6, 12, 24, 48, 96, 168]:

    col = f"generation_lag_{lag:03d}h"

    expected_values = original_generation.shift(lag)

    assert df[col].equals(expected_values), (
        f"Falha de alinhamento no lag {lag} h."
    )

assert df["generation_mw"].equals(
    original_generation
), "A geração original foi alterada."

assert (
    df["co2_metric_tons"].notna().sum() == 6249
), "A variável-alvo foi alterada."

# ------------------------------------------------------------
# 5. DISPONIBILIDADE DOS LAGS
# ------------------------------------------------------------

eligible = df["eligible_for_modeling"]

availability = []

for lag, col in enumerate(lag_columns, start=1):

    available = int(
        (eligible & df[col].notna()).sum()
    )

    missing = int(
        (eligible & df[col].isna()).sum()
    )

    availability.append({
        "lag_hours": lag,
        "column": col,
        "eligible_hours": int(eligible.sum()),
        "available_hours": available,
        "missing_hours": missing,
        "availability_pct": round(
            100 * available / eligible.sum(), 4
        ),
    })

availability_df = pd.DataFrame(availability)

_project_print("\nDISPONIBILIDADE DOS LAGS")
_project_print(
    availability_df[
        availability_df["lag_hours"].isin(
            [1, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168]
        )
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. DISPONIBILIDADE CONJUNTA POR HORIZONTE
# ------------------------------------------------------------
#
# Para um horizonte k, verifica se todos os lags
# de 1 até k estão disponíveis.
#
# Não exclui linhas do arquivo original.
# ------------------------------------------------------------

HORIZONS = [
    0, 1, 3, 6, 12, 24, 36,
    48, 72, 96, 120, 144, 168
]

horizon_summary = []

for horizon in HORIZONS:

    if horizon == 0:
        complete = eligible & df["generation_mw"].notna()
    else:
        cols = lag_columns[:horizon]
        complete = (
            eligible
            & df["generation_mw"].notna()
            & df[cols].notna().all(axis=1)
        )

    row = {
        "horizon_hours": horizon,
        "eligible_complete_hours": int(complete.sum()),
    }

    for part in ["train", "validation", "test"]:
        row[f"{part}_hours"] = int(
            (complete & (df["partition"] == part)).sum()
        )

    horizon_summary.append(row)

horizon_df = pd.DataFrame(horizon_summary)

_project_print("\nAMOSTRA COMPLETA POR HORIZONTE")
_project_print(horizon_df.to_string(index=False))

# ------------------------------------------------------------
# 7. REGISTRO DA AUDITORIA
# ------------------------------------------------------------

report = {
    "cell": "04/14",
    "source": str(INPUT_PATH),
    "output": str(OUTPUT_DATA),
    "calendar_hours": int(len(df)),
    "eligible_hours": int(eligible.sum()),
    "max_lag_hours": MAX_LAG,
    "lag_count": len(lag_columns),
    "lag_method": "calendar_hour_shift",
    "missing_values_imputed": False,
    "zero_generation_preserved": True,
    "co2_lags_created": False,
    "process_features_created": False,
    "partition_counts": partition_counts,
    "horizon_summary": horizon_summary,
    "availability_selected_lags": (
        availability_df[
            availability_df["lag_hours"].isin(
                [1, 3, 6, 12, 24, 48, 72, 96, 168]
            )
        ].to_dict(orient="records")
    ),
}

# ------------------------------------------------------------
# 8. EXPORTAÇÃO
# ------------------------------------------------------------

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df.to_parquet(OUTPUT_DATA, index=False)

_project_csv(availability_df, 
    OUTPUT_AVAILABILITY, index=False
)

with open(OUTPUT_REPORT, "w", encoding="utf-8") as f:
    _project_dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 9. INTEGRIDADE DOS ARQUIVOS
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

for path in [
    OUTPUT_DATA,
    OUTPUT_AVAILABILITY,
    OUTPUT_REPORT,
]:
    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 04/14 CONCLUÍDA — "
    "BANCO DE LAGS DE GERAÇÃO"
)
_project_print("=" * 72)

_project_print("Horas preservadas:", len(df))
_project_print("Lags construídos:", len(lag_columns))
_project_print("Horizonte máximo:", MAX_LAG, "h")
_project_print("Nenhum modelo treinado.")
_project_print(
    "PRÓXIMA CÉLULA: 05 — "
    "diagnóstico estatístico das séries temporais."
)


In [ ]:
# Referência metodológica: statsmodels PACF. ACF/CCF por correlação pareada e PACF no trecho contíguo são diagnósticos, não regras de seleção.

# ============================================================
# CÉLULA 05/14 — DIAGNÓSTICO ESTATÍSTICO DOS LAGS
# ============================================================
#
# PROJETO: Seleção preditiva de lags de geração para CO2
# UNIDADE: Iatan Unit 2 | ORISPL 6065 | 2025
#
# OBJETIVOS:
# 1. Carregar o banco de lags da Célula 04.
# 2. Calcular ACF e PACF da geração.
# 3. Calcular correlação entre G(t-k) e CO2(t).
# 4. Produzir gráficos e tabelas descritivas.
# 5. Registrar os resultados sem selecionar lags.
#
# REGRAS:
# - Somente partição de treinamento.
# - Lags em horas reais de calendário.
# - Sem imputação de valores ausentes.
# - Sem CO2 defasado como preditor.
# - Sem treinamento de modelos.
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import acf, pacf

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

OUTPUT_TABLE = (
    REPORT_DIR / "cell05_lag_statistical_diagnostics.csv"
)

OUTPUT_REPORT = (
    REPORT_DIR / "cell05_statistical_diagnostics.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell05_acf_pacf_ccf.png"
)

MAX_LAG = 168

assert INPUT_PATH.exists(), (
    f"Arquivo não encontrado: {INPUT_PATH}"
)

# ------------------------------------------------------------
# 2. CARREGAMENTO E VERIFICAÇÕES
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_PATH).copy()

required = [
    "timestamp",
    "generation_mw",
    "co2_metric_tons",
    "eligible_for_modeling",
    "partition",
]

lag_columns = [
    f"generation_lag_{k:03d}h"
    for k in range(1, MAX_LAG + 1)
]

missing_columns = [
    c for c in required + lag_columns
    if c not in df.columns
]

assert not missing_columns, (
    f"Colunas ausentes: {missing_columns}"
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="raise"
)

df = df.sort_values("timestamp").reset_index(drop=True)

assert len(df) == 8760
assert df["timestamp"].is_unique
assert int(df["eligible_for_modeling"].sum()) == 6249

train_mask = df["partition"].eq("train")

assert int(train_mask.sum()) == 4499

train_end = df.loc[
    train_mask, "timestamp"
].max()

# Selecionar o período cronológico de treinamento.
# Isso mantém as horas ausentes entre as horas elegíveis.

train_calendar = df.loc[
    df["timestamp"] <= train_end
].copy()

# ACF/PACF: geração observada no calendário de treino.
generation_train = train_calendar[
    "generation_mw"
].astype(float)

# Correlação G(t-k) versus CO2(t):
# somente linhas elegíveis de treinamento.
train = df.loc[
    train_mask & df["eligible_for_modeling"]
].copy()

_project_print("Fonte:", INPUT_PATH)
_project_print("Fim do treinamento:", train_end)
_project_print("Horas no calendário de treino:", len(train_calendar))
_project_print("Horas elegíveis de treino:", len(train))
_project_print("Lags avaliados:", MAX_LAG)

# ------------------------------------------------------------
# 3. ACF DA GERAÇÃO
# ------------------------------------------------------------
#
# Calculada por pares observados no calendário.
# Valores ausentes não são convertidos em zero.
#
# Para cada k, correlacionamos G(t) e G(t-k)
# quando ambos estão disponíveis.
# ------------------------------------------------------------

acf_values = []
acf_pairs = []

for k in range(MAX_LAG + 1):

    current = generation_train

    if k == 0:
        previous = generation_train
    else:
        previous = generation_train.shift(k)

    valid = current.notna() & previous.notna()

    n_pairs = int(valid.sum())

    if n_pairs >= 3:
        x = current.loc[valid].to_numpy()
        y = previous.loc[valid].to_numpy()

        if np.std(x) > 0 and np.std(y) > 0:
            corr = float(np.corrcoef(x, y)[0, 1])
        else:
            corr = np.nan
    else:
        corr = np.nan

    acf_values.append(corr)
    acf_pairs.append(n_pairs)

# ------------------------------------------------------------
# 4. PACF DA GERAÇÃO
# ------------------------------------------------------------
#
# A PACF convencional exige uma série regular sem NaN.
# Para evitar imputação, usamos o maior trecho contínuo
# com geração observada dentro do treinamento.
#
# A PACF é exploratória e não determina a seleção final.
# ------------------------------------------------------------

observed = generation_train.notna()

group_id = observed.ne(
    observed.shift(fill_value=False)
).cumsum()

segments = [
    group.astype(float).to_numpy()
    for _, group in generation_train[
        observed
    ].groupby(group_id[observed])
]

longest_segment = max(
    segments,
    key=len
)

pacf_max_lag = min(
    MAX_LAG,
    len(longest_segment) // 2 - 1
)

assert pacf_max_lag >= 1, (
    "Trecho contínuo insuficiente para PACF."
)

pacf_values = pacf(
    longest_segment,
    nlags=pacf_max_lag,
    method="ywm"
)

pacf_full = np.full(
    MAX_LAG + 1,
    np.nan,
    dtype=float
)

pacf_full[:len(pacf_values)] = pacf_values

_project_print(
    "\nMaior trecho contínuo para PACF:",
    len(longest_segment),
    "horas"
)
_project_print("PACF calculada até:", pacf_max_lag, "h")

# ------------------------------------------------------------
# 5. CORRELAÇÃO CRUZADA G(t-k) x CO2(t)
# ------------------------------------------------------------
#
# Esta análise não utiliza CO2 futuro nem CO2 defasado.
# O alvo é CO2(t), apenas nas horas de treinamento.
#
# Cada lag é avaliado com os pares disponíveis.
# ------------------------------------------------------------

target = train["co2_metric_tons"].astype(float)

ccf_values = []
ccf_pairs = []

for k in range(MAX_LAG + 1):

    if k == 0:
        predictor = train["generation_mw"].astype(float)
    else:
        predictor = train[
            f"generation_lag_{k:03d}h"
        ].astype(float)

    valid = predictor.notna() & target.notna()

    n_pairs = int(valid.sum())

    if n_pairs >= 3:
        x = predictor.loc[valid].to_numpy()
        y = target.loc[valid].to_numpy()

        if np.std(x) > 0 and np.std(y) > 0:
            corr = float(np.corrcoef(x, y)[0, 1])
        else:
            corr = np.nan
    else:
        corr = np.nan

    ccf_values.append(corr)
    ccf_pairs.append(n_pairs)

# ------------------------------------------------------------
# 6. TABELA CONSOLIDADA
# ------------------------------------------------------------

diagnostics = pd.DataFrame({
    "lag_hours": np.arange(MAX_LAG + 1),
    "acf_generation": acf_values,
    "acf_pairs": acf_pairs,
    "pacf_generation": pacf_full,
    "ccf_generation_co2": ccf_values,
    "ccf_pairs": ccf_pairs,
})

diagnostics["abs_ccf"] = (
    diagnostics["ccf_generation_co2"].abs()
)

_project_print("\nDIAGNÓSTICOS — LAGS SELECIONADOS")
_project_print(
    diagnostics.loc[
        diagnostics["lag_hours"].isin(
            [0, 1, 3, 6, 12, 24, 36,
             48, 72, 96, 120, 144, 168]
        )
    ].to_string(index=False)
)

_project_print("\nMAIORES CORRELAÇÕES ABSOLUTAS COM CO2")
_project_print(
    diagnostics.loc[
        diagnostics["lag_hours"] > 0
    ].nlargest(
        15, "abs_ccf"
    )[
        [
            "lag_hours",
            "ccf_generation_co2",
            "ccf_pairs",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 7. GRÁFICOS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    3, 1,
    figsize=(13, 11),
    sharex=True
)

lags = diagnostics["lag_hours"].to_numpy()

axes[0].plot(
    lags,
    diagnostics["acf_generation"],
    linewidth=1.4
)
axes[0].axhline(0, linewidth=0.8)
axes[0].set_ylabel("ACF")
axes[0].set_title(
    "Autocorrelação da geração — treinamento"
)

axes[1].plot(
    lags,
    diagnostics["pacf_generation"],
    linewidth=1.4
)
axes[1].axhline(0, linewidth=0.8)
axes[1].set_ylabel("PACF")
axes[1].set_title(
    "Autocorrelação parcial — maior trecho contínuo"
)

axes[2].plot(
    lags,
    diagnostics["ccf_generation_co2"],
    linewidth=1.4
)
axes[2].axhline(0, linewidth=0.8)
axes[2].set_ylabel("Correlação")
axes[2].set_xlabel("Lag da geração (horas)")
axes[2].set_title(
    "Correlação G(t-k) versus CO2(t) — treinamento"
)

for ax in axes:
    ax.grid(alpha=0.25)
    ax.set_xlim(0, MAX_LAG)

plt.tight_layout()
plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 8. EXPORTAÇÃO E RASTREABILIDADE
# ------------------------------------------------------------

_project_csv(diagnostics, 
    OUTPUT_TABLE,
    index=False
)

report = {
    "cell": "05/14",
    "source": str(INPUT_PATH),
    "train_end": str(train_end),
    "train_calendar_hours": int(len(train_calendar)),
    "train_eligible_hours": int(len(train)),
    "max_lag_hours": MAX_LAG,
    "acf_method": "pairwise_calendar_correlation",
    "pacf_method": "ywm_longest_contiguous_observed_segment",
    "pacf_segment_hours": int(len(longest_segment)),
    "pacf_max_lag": int(pacf_max_lag),
    "ccf_method": "pairwise_generation_lag_vs_current_co2",
    "missing_imputation": False,
    "validation_used": False,
    "test_used": False,
    "lags_selected": False,
    "models_trained": False,
    "output_table": str(OUTPUT_TABLE),
    "output_figure": str(OUTPUT_FIGURE),
}

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

for path in [
    OUTPUT_TABLE,
    OUTPUT_REPORT,
    OUTPUT_FIGURE,
]:
    assert path.exists()
    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 05/14 CONCLUÍDA — "
    "DIAGNÓSTICO ESTATÍSTICO"
)
_project_print("=" * 72)
_project_print("Lags examinados:", MAX_LAG)
_project_print("Nenhum lag selecionado.")
_project_print("Nenhum modelo treinado.")
_project_print(
    "PRÓXIMA CÉLULA: 06 — "
    "baseline com Ridge e Random Forest."
)


In [ ]:
# Referência metodológica: scikit-learn Ridge e Breiman (2001). Hyndman e Koehler (2006) para interpretação de RMSE e MAE.

# ============================================================
# CÉLULA 06/14 — BASELINE COM RIDGE E RANDOM FOREST
# ============================================================
#
# PROJETO: Seleção preditiva de lags de geração para CO2
# UNIDADE: Iatan Unit 2 | ORISPL 6065 | Ano 2025
#
# OBJETIVOS:
# 1. Carregar os dados aprovados da Célula 04.
# 2. Treinar Ridge e Random Forest usando somente G(t).
# 3. Avaliar os modelos na partição de validação.
# 4. Calcular RMSE, MAE e R².
# 5. Salvar modelos, previsões, métricas e protocolo.
#
# REGRAS:
# - Treinamento: 4.499 observações.
# - Validação: 687 observações.
# - Teste: reservado para a Célula 13.
# - Nenhum lag utilizado nesta etapa.
# - Nenhuma variável de processos.
# - Nenhuma informação de CO2 futuro como preditor.
# ============================================================

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"
MODEL_DIR = RUN_DIR / "models"

for directory in [REPORT_DIR, FIG_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

OUTPUT_METRICS = (
    REPORT_DIR / "cell06_baseline_metrics.csv"
)

OUTPUT_PREDICTIONS = (
    DATA_DIR / "cell06_baseline_validation_predictions.parquet"
)

OUTPUT_REPORT = (
    REPORT_DIR / "cell06_baseline_protocol.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell06_baseline_validation.png"
)

assert INPUT_PATH.exists(), (
    f"Arquivo não encontrado: {INPUT_PATH}"
)

RANDOM_STATE = 42

# Parâmetros fixos para o experimento inicial.
# Sem ajuste por teste e sem busca de hiperparâmetros.

RIDGE_ALPHA = 1.0

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 3,
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# ------------------------------------------------------------
# 2. CARREGAMENTO E VERIFICAÇÃO
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_PATH)

required = [
    "timestamp",
    "generation_mw",
    "co2_metric_tons",
    "eligible_for_modeling",
    "partition",
]

missing = [c for c in required if c not in df.columns]

assert not missing, (
    f"Colunas ausentes: {missing}"
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="raise"
)

assert len(df) == 8760
assert df["timestamp"].is_unique
assert int(df["eligible_for_modeling"].sum()) == 6249

train = df.loc[
    (df["partition"] == "train")
    & df["eligible_for_modeling"]
].copy()

validation = df.loc[
    (df["partition"] == "validation")
    & df["eligible_for_modeling"]
].copy()

assert len(train) == 4499
assert len(validation) == 687

assert train["timestamp"].max() < (
    validation["timestamp"].min()
)

FEATURES = ["generation_mw"]
TARGET = "co2_metric_tons"

assert train[FEATURES + [TARGET]].notna().all().all()
assert validation[FEATURES + [TARGET]].notna().all().all()

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = validation[FEATURES]
y_val = validation[TARGET]

_project_print("Fonte:", INPUT_PATH)
_project_print("Variável preditora:", FEATURES)
_project_print("Variável-alvo:", TARGET)
_project_print("Treino:", len(train))
_project_print("Validação:", len(validation))
_project_print("Teste: reservado, não utilizado")

# ------------------------------------------------------------
# 3. DEFINIÇÃO DOS MODELOS
# ------------------------------------------------------------

models = {
    "ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=RIDGE_ALPHA)),
    ]),
    "random_forest": RandomForestRegressor(
        **RF_PARAMS
    ),
}

# ------------------------------------------------------------
# 4. TREINAMENTO E VALIDAÇÃO
# ------------------------------------------------------------

metrics_rows = []

predictions = validation[
    ["timestamp", "generation_mw", TARGET]
].copy()

predictions = predictions.rename(
    columns={TARGET: "co2_observed_metric_tons"}
)

for name, model in models.items():

    _project_print(f"\nTreinando: {name}")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    rmse = float(
        np.sqrt(
            mean_squared_error(y_val, y_pred)
        )
    )

    mae = float(
        mean_absolute_error(y_val, y_pred)
    )

    r2 = float(
        r2_score(y_val, y_pred)
    )

    metrics_rows.append({
        "model": name,
        "feature_set": "generation_current_only",
        "n_features": 1,
        "train_rows": len(train),
        "validation_rows": len(validation),
        "rmse_metric_tons": rmse,
        "mae_metric_tons": mae,
        "r2": r2,
    })

    predictions[
        f"co2_pred_{name}_metric_tons"
    ] = y_pred

    model_path = (
        MODEL_DIR / f"cell06_baseline_{name}.joblib"
    )

    joblib.dump(model, model_path)

    _project_print(f"RMSE: {rmse:.4f} toneladas métricas")
    _project_print(f"MAE:  {mae:.4f} toneladas métricas")
    _project_print(f"R²:   {r2:.6f}")

metrics_df = pd.DataFrame(metrics_rows)

metrics_df = metrics_df.sort_values(
    "rmse_metric_tons"
).reset_index(drop=True)

_project_print("\nRESULTADOS DO BASELINE")
_project_print(metrics_df.to_string(index=False))

# ------------------------------------------------------------
# 5. GRÁFICOS DE VALIDAÇÃO
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5)
)

observed = predictions[
    "co2_observed_metric_tons"
].to_numpy()

minimum = float(np.min(observed))
maximum = float(np.max(observed))

for ax, name in zip(
    axes,
    ["ridge", "random_forest"]
):

    predicted = predictions[
        f"co2_pred_{name}_metric_tons"
    ].to_numpy()

    ax.scatter(
        observed,
        predicted,
        alpha=0.35,
        s=18
    )

    ax.plot(
        [minimum, maximum],
        [minimum, maximum],
        linestyle="--",
        linewidth=1.2
    )

    ax.set_title(name)
    ax.set_xlabel(
        "CO2 observado (toneladas métricas)"
    )
    ax.set_ylabel(
        "CO2 estimado (toneladas métricas)"
    )
    ax.grid(alpha=0.2)

plt.suptitle(
    "Baseline G(t) — Validação cronológica"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. EXPORTAÇÃO
# ------------------------------------------------------------

_project_csv(metrics_df, 
    OUTPUT_METRICS,
    index=False
)

predictions.to_parquet(
    OUTPUT_PREDICTIONS,
    index=False
)

report = {
    "cell": "06/14",
    "project": "predictive_lag_selection",
    "source": str(INPUT_PATH),
    "features": FEATURES,
    "target": TARGET,
    "target_unit": "metric_tons_per_hour",
    "train_rows": int(len(train)),
    "validation_rows": int(len(validation)),
    "test_used": False,
    "ridge_alpha": RIDGE_ALPHA,
    "random_forest_parameters": RF_PARAMS,
    "random_state": RANDOM_STATE,
    "models": list(models.keys()),
    "metrics": metrics_rows,
    "lag_features_used": False,
    "process_features_used": False,
    "output_metrics": str(OUTPUT_METRICS),
    "output_predictions": str(OUTPUT_PREDICTIONS),
}

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 7. VERIFICAÇÃO DE INTEGRIDADE
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

files_to_verify = [
    OUTPUT_METRICS,
    OUTPUT_PREDICTIONS,
    OUTPUT_REPORT,
    OUTPUT_FIGURE,
    MODEL_DIR / "cell06_baseline_ridge.joblib",
    MODEL_DIR / "cell06_baseline_random_forest.joblib",
]

for path in files_to_verify:

    assert path.exists(), (
        f"Arquivo não encontrado: {path}"
    )

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 06/14 CONCLUÍDA — "
    "BASELINE PREDITIVO"
)
_project_print("=" * 72)

_project_print("Preditores utilizados: G(t)")
_project_print("Treino:", len(train))
_project_print("Validação:", len(validation))
_project_print("Teste preservado.")
_project_print("Nenhum lag utilizado.")
_project_print(
    "PRÓXIMA CÉLULA: 07 — "
    "comparação dos horizontes de lags."
)


In [ ]:
# Referência metodológica: Cerqueira et al. (2020) e Hyndman e Koehler (2006). Horizonte e cobertura comuns na comparação temporal.

# ============================================================
# CÉLULA 07/14 — COMPARAÇÃO DOS HORIZONTES DE LAGS
# ============================================================
#
# PROJETO: Seleção preditiva de lags para estimação de CO2
# UNIDADE: Iatan Unit 2 | ORISPL 6065 | Ano 2025
#
# OBJETIVOS:
# 1. Comparar horizontes cumulativos de 0 a 168 horas.
# 2. Treinar Ridge e Random Forest.
# 3. Comparar cada horizonte com baseline na mesma amostra.
# 4. Medir ganho preditivo de RMSE, MAE e R2.
# 5. Salvar resultados e gráficos.
#
# REGRAS:
# - Apenas geração atual e seus lags.
# - Treino e validação cronológicos já definidos.
# - Teste reservado para a Célula 13.
# - Nenhuma imputação de valores ausentes.
# - Nenhuma variável de processos.
# - Sem seleção definitiva de lags nesta célula.
# ============================================================

from pathlib import Path
import json
import hashlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

BASELINE_PATH = (
    REPORT_DIR / "cell06_baseline_metrics.csv"
)

OUTPUT_RESULTS = (
    REPORT_DIR / "cell07_horizon_comparison.csv"
)

OUTPUT_REPORT = (
    REPORT_DIR / "cell07_horizon_comparison.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell07_horizon_rmse_comparison.png"
)

OUTPUT_GAIN_FIGURE = (
    FIG_DIR / "cell07_horizon_relative_gain.png"
)

HORIZONS = [
    0, 1, 3, 6, 12, 24, 36,
    48, 72, 96, 120, 144, 168
]

RANDOM_STATE = 42
RIDGE_ALPHA = 1.0

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 3,
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

TARGET = "co2_metric_tons"
CURRENT_FEATURE = "generation_mw"

assert INPUT_PATH.exists(), (
    f"Arquivo não encontrado: {INPUT_PATH}"
)

assert BASELINE_PATH.exists(), (
    "Execute a Célula 06 antes da Célula 07."
)

# ------------------------------------------------------------
# 2. CARREGAMENTO E VALIDAÇÃO
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_PATH)

required = [
    "timestamp",
    CURRENT_FEATURE,
    TARGET,
    "eligible_for_modeling",
    "partition",
]

lag_columns = [
    f"generation_lag_{k:03d}h"
    for k in range(1, max(HORIZONS) + 1)
]

missing = [
    c for c in required + lag_columns
    if c not in df.columns
]

assert not missing, (
    f"Colunas ausentes: {missing}"
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="raise"
)

df = df.sort_values("timestamp").reset_index(drop=True)

assert len(df) == 8760
assert df["timestamp"].is_unique
assert int(df["eligible_for_modeling"].sum()) == 6249

assert (
    (df["partition"] == "train").sum() == 4499
)

assert (
    (df["partition"] == "validation").sum() == 687
)

baseline_reference = pd.read_csv(BASELINE_PATH)

_project_print("Fonte:", INPUT_PATH)
_project_print("Horizontes:", HORIZONS)
_project_print("Modelos: Ridge e Random Forest")
_project_print("Teste: reservado")

# ------------------------------------------------------------
# 3. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def build_model(name):

    if name == "ridge":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=RIDGE_ALPHA)),
        ])

    if name == "random_forest":
        return RandomForestRegressor(
            **RF_PARAMS
        )

    raise ValueError(f"Modelo desconhecido: {name}")


def calculate_metrics(y_true, y_pred):

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
    }


def fit_and_evaluate(
    model_name,
    features,
    train_data,
    validation_data
):

    model = build_model(model_name)

    model.fit(
        train_data[features],
        train_data[TARGET]
    )

    predictions = model.predict(
        validation_data[features]
    )

    return calculate_metrics(
        validation_data[TARGET],
        predictions
    )

# ------------------------------------------------------------
# 4. EXPERIMENTO POR HORIZONTE
# ------------------------------------------------------------
#
# Para cada horizonte:
#
# H = 0:
#   G(t)
#
# H = k:
#   G(t), G(t-1), ..., G(t-k)
#
# Cada horizonte utiliza apenas observações completas
# para suas respectivas variáveis.
#
# Baseline e modelo com lags usam exatamente as mesmas
# linhas de treino e validação dentro de cada horizonte.
# ------------------------------------------------------------

results = []

start_time = time.time()

for horizon in HORIZONS:

    features = [CURRENT_FEATURE] + [
        f"generation_lag_{k:03d}h"
        for k in range(1, horizon + 1)
    ]

    valid_rows = (
        df["eligible_for_modeling"]
        & df[features + [TARGET]].notna().all(axis=1)
    )

    train_data = df.loc[
        valid_rows
        & (df["partition"] == "train")
    ].copy()

    validation_data = df.loc[
        valid_rows
        & (df["partition"] == "validation")
    ].copy()

    n_train = len(train_data)
    n_val = len(validation_data)

    assert n_train >= 100, (
        f"Treino insuficiente para H={horizon}"
    )

    assert n_val >= 30, (
        f"Validação insuficiente para H={horizon}"
    )

    _project_print(
        f"\nHorizonte {horizon:3d} h | "
        f"Treino: {n_train} | "
        f"Validação: {n_val}"
    )

    for model_name in ["ridge", "random_forest"]:

        # Baseline G(t) na mesma amostra
        baseline_metrics = fit_and_evaluate(
            model_name,
            [CURRENT_FEATURE],
            train_data,
            validation_data
        )

        # Modelo com horizonte acumulado
        horizon_metrics = fit_and_evaluate(
            model_name,
            features,
            train_data,
            validation_data
        )

        gain_rmse_pct = (
            100
            * (
                baseline_metrics["rmse"]
                - horizon_metrics["rmse"]
            )
            / baseline_metrics["rmse"]
        )

        gain_mae_pct = (
            100
            * (
                baseline_metrics["mae"]
                - horizon_metrics["mae"]
            )
            / baseline_metrics["mae"]
        )

        results.append({
            "model": model_name,
            "horizon_hours": horizon,
            "n_features": len(features),
            "train_rows": n_train,
            "validation_rows": n_val,
            "baseline_rmse": baseline_metrics["rmse"],
            "horizon_rmse": horizon_metrics["rmse"],
            "rmse_gain_pct": gain_rmse_pct,
            "baseline_mae": baseline_metrics["mae"],
            "horizon_mae": horizon_metrics["mae"],
            "mae_gain_pct": gain_mae_pct,
            "baseline_r2": baseline_metrics["r2"],
            "horizon_r2": horizon_metrics["r2"],
        })

        _project_print(
            f"  {model_name:13s} | "
            f"RMSE baseline: "
            f"{baseline_metrics['rmse']:.3f} | "
            f"RMSE lags: "
            f"{horizon_metrics['rmse']:.3f} | "
            f"Ganho: {gain_rmse_pct:+.2f}%"
        )

elapsed = time.time() - start_time

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 5. AUDITORIA DO EXPERIMENTO
# ------------------------------------------------------------

assert len(results_df) == (
    len(HORIZONS) * 2
)

assert results_df[
    [
        "baseline_rmse",
        "horizon_rmse",
        "baseline_mae",
        "horizon_mae",
    ]
].notna().all().all()

# H=0 deve reproduzir o baseline da Célula 06.
for model_name in ["ridge", "random_forest"]:

    current = results_df.loc[
        (results_df["model"] == model_name)
        & (results_df["horizon_hours"] == 0)
    ].iloc[0]

    reference = baseline_reference.loc[
        baseline_reference["model"] == model_name
    ].iloc[0]

    assert np.isclose(
        current["horizon_rmse"],
        reference["rmse_metric_tons"],
        rtol=1e-8,
        atol=1e-8
    ), (
        f"Baseline H=0 divergente: {model_name}"
    )

_project_print("\nCOMPARAÇÃO COMPLETA")
_project_print(
    results_df[
        [
            "model",
            "horizon_hours",
            "n_features",
            "train_rows",
            "validation_rows",
            "baseline_rmse",
            "horizon_rmse",
            "rmse_gain_pct",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. MELHORES RESULTADOS EXPLORATÓRIOS
# ------------------------------------------------------------
#
# Ranking preliminar, não seleção definitiva.
# Os horizontes possuem amostras diferentes.
# ------------------------------------------------------------

_project_print("\nMAIOR GANHO RELATIVO POR MODELO")

for model_name in ["ridge", "random_forest"]:

    subset = results_df.loc[
        results_df["model"] == model_name
    ]

    best = subset.loc[
        subset["rmse_gain_pct"].idxmax()
    ]

    _project_print(
        f"{model_name}: "
        f"H={int(best['horizon_hours'])} h | "
        f"Ganho={best['rmse_gain_pct']:+.2f}% | "
        f"RMSE={best['horizon_rmse']:.3f}"
    )

# ------------------------------------------------------------
# 7. GRÁFICO DE RMSE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 6))

for model_name in ["ridge", "random_forest"]:

    subset = results_df.loc[
        results_df["model"] == model_name
    ].sort_values("horizon_hours")

    ax.plot(
        subset["horizon_hours"],
        subset["horizon_rmse"],
        marker="o",
        label=f"{model_name} — com lags"
    )

    ax.plot(
        subset["horizon_hours"],
        subset["baseline_rmse"],
        linestyle="--",
        alpha=0.7,
        label=f"{model_name} — baseline pareado"
    )

ax.set_xlabel("Horizonte acumulado (horas)")
ax.set_ylabel("RMSE (toneladas métricas/h)")
ax.set_title(
    "RMSE por horizonte — validação cronológica"
)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 8. GRÁFICO DE GANHO RELATIVO
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5))

for model_name in ["ridge", "random_forest"]:

    subset = results_df.loc[
        results_df["model"] == model_name
    ].sort_values("horizon_hours")

    ax.plot(
        subset["horizon_hours"],
        subset["rmse_gain_pct"],
        marker="o",
        label=model_name
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Horizonte acumulado (horas)")
ax.set_ylabel("Redução do RMSE (%)")
ax.set_title(
    "Ganho preditivo dos lags sobre baseline pareado"
)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.savefig(
    OUTPUT_GAIN_FIGURE,
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 9. EXPORTAÇÃO
# ------------------------------------------------------------

_project_csv(results_df, 
    OUTPUT_RESULTS,
    index=False
)

report = {
    "cell": "07/14",
    "project": "predictive_lag_selection",
    "source": str(INPUT_PATH),
    "baseline_reference": str(BASELINE_PATH),
    "horizons": HORIZONS,
    "models": ["ridge", "random_forest"],
    "ridge_alpha": RIDGE_ALPHA,
    "random_forest_parameters": RF_PARAMS,
    "random_state": RANDOM_STATE,
    "comparison_method": (
        "paired_baseline_within_each_horizon"
    ),
    "train_validation_method": (
        "fixed_chronological_partitions"
    ),
    "missing_imputation": False,
    "test_used": False,
    "process_features_used": False,
    "co2_lags_used": False,
    "lags_finally_selected": False,
    "execution_seconds": round(elapsed, 2),
    "results": results_df.to_dict(
        orient="records"
    ),
}

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 10. VERIFICAÇÃO DE INTEGRIDADE
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


for path in [
    OUTPUT_RESULTS,
    OUTPUT_REPORT,
    OUTPUT_FIGURE,
    OUTPUT_GAIN_FIGURE,
]:

    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 07/14 CONCLUÍDA — "
    "COMPARAÇÃO DOS HORIZONTES"
)
_project_print("=" * 72)

_project_print("Horizontes avaliados:", len(HORIZONS))
_project_print("Modelos:", 2)
_project_print("Comparações:", len(results_df))
_project_print(f"Tempo de execução: {elapsed:.1f} s")
_project_print("Teste preservado.")
_project_print("Nenhum lag selecionado definitivamente.")
_project_print(
    "PRÓXIMA CÉLULA: 08 — "
    "importância individual e redundância dos lags."
)


In [ ]:
# Referência metodológica: scikit-learn permutation_importance. Interprete a importância junto à redundância entre preditores correlacionados.

# ============================================================
# CÉLULA 08/14 — IMPORTÂNCIA E REDUNDÂNCIA DOS LAGS
# ============================================================
#
# PROJETO: Seleção preditiva de lags de geração para CO2
# UNIDADE: Iatan Unit 2 | ORISPL 6065 | Ano 2025
#
# OBJETIVOS:
# 1. Avaliar importância dos lags de geração no RF.
# 2. Medir contribuição por permutação na validação.
# 3. Investigar redundância entre defasagens.
# 4. Identificar candidatos para a Célula 09.
#
# REGRAS:
# - Somente geração atual e seus lags.
# - Horizonte exploratório: 72 horas.
# - Partições cronológicas preservadas.
# - Teste não utilizado.
# - Sem imputação.
# - Sem seleção definitiva de lags.
# ============================================================

from pathlib import Path
import json
import hashlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

CELL07_PATH = (
    REPORT_DIR / "cell07_horizon_comparison.csv"
)

OUTPUT_IMPORTANCE = (
    REPORT_DIR / "cell08_lag_importance.csv"
)

OUTPUT_REDUNDANCY = (
    REPORT_DIR / "cell08_lag_redundancy.csv"
)

OUTPUT_REPORT = (
    REPORT_DIR / "cell08_importance_redundancy.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell08_lag_importance.png"
)

OUTPUT_HEATMAP = (
    FIG_DIR / "cell08_lag_redundancy_heatmap.png"
)

HORIZON = 72
RANDOM_STATE = 42
PERMUTATION_REPEATS = 5

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 3,
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

assert INPUT_PATH.exists()
assert CELL07_PATH.exists()

# ------------------------------------------------------------
# 2. CARREGAMENTO E AUDITORIA
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_PATH)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="raise"
)

FEATURES = ["generation_mw"] + [
    f"generation_lag_{k:03d}h"
    for k in range(1, HORIZON + 1)
]

TARGET = "co2_metric_tons"

required = [
    "timestamp",
    "partition",
    "eligible_for_modeling",
    TARGET,
] + FEATURES

missing = [
    c for c in required
    if c not in df.columns
]

assert not missing, f"Colunas ausentes: {missing}"

assert len(df) == 8760
assert df["timestamp"].is_unique
assert int(df["eligible_for_modeling"].sum()) == 6249

valid_rows = (
    df["eligible_for_modeling"]
    & df[FEATURES + [TARGET]].notna().all(axis=1)
)

train = df.loc[
    valid_rows & df["partition"].eq("train")
].copy()

validation = df.loc[
    valid_rows & df["partition"].eq("validation")
].copy()

assert len(train) == 3025
assert len(validation) == 384

assert (
    train["timestamp"].max()
    < validation["timestamp"].min()
)

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = validation[FEATURES]
y_val = validation[TARGET]

_project_print("Fonte:", INPUT_PATH)
_project_print("Horizonte exploratório:", HORIZON, "h")
_project_print("Preditores:", len(FEATURES))
_project_print("Treino:", len(train))
_project_print("Validação:", len(validation))
_project_print("Teste: reservado")

# ------------------------------------------------------------
# 3. TREINAMENTO DO MODELO EXPLORATÓRIO
# ------------------------------------------------------------

start_time = time.time()

rf = RandomForestRegressor(**RF_PARAMS)

rf.fit(X_train, y_train)

pred = rf.predict(X_val)

rmse = float(
    np.sqrt(mean_squared_error(y_val, pred))
)

_project_print(f"\nRMSE RF (72 h): {rmse:.6f}")

# Conferência com a Célula 07.
cell07 = pd.read_csv(CELL07_PATH)

reference = cell07.loc[
    cell07["model"].eq("random_forest")
    & cell07["horizon_hours"].eq(HORIZON)
].iloc[0]

assert np.isclose(
    rmse,
    reference["horizon_rmse"],
    rtol=1e-8,
    atol=1e-8
), "Resultado divergente da Célula 07."

# ------------------------------------------------------------
# 4. IMPORTÂNCIA POR IMPUREZA
# ------------------------------------------------------------
#
# Importância interna do Random Forest.
# Pode favorecer variáveis correlacionadas ou contínuas.
# Será interpretada apenas como diagnóstico auxiliar.
# ------------------------------------------------------------

impurity_importance = rf.feature_importances_

# ------------------------------------------------------------
# 5. IMPORTÂNCIA POR PERMUTAÇÃO
# ------------------------------------------------------------
#
# Mede o aumento do erro quando cada variável é
# embaralhada individualmente na validação.
#
# Como os lags são correlacionados, a importância
# individual pode ser subestimada por redundância.
#
# Não representa causalidade.
# ------------------------------------------------------------

permutation = permutation_importance(
    rf,
    X_val,
    y_val,
    scoring="neg_root_mean_squared_error",
    n_repeats=PERMUTATION_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "lag_hours": list(range(HORIZON + 1)),
    "impurity_importance": impurity_importance,
    "permutation_rmse_increase_mean": (
        permutation.importances_mean
    ),
    "permutation_rmse_increase_std": (
        permutation.importances_std
    ),
})

importance_df = importance_df.sort_values(
    "permutation_rmse_increase_mean",
    ascending=False
).reset_index(drop=True)

importance_df["rank_permutation"] = (
    np.arange(1, len(importance_df) + 1)
)

_project_print("\nTOP 20 — IMPORTÂNCIA POR PERMUTAÇÃO")
_project_print(
    importance_df[
        [
            "lag_hours",
            "permutation_rmse_increase_mean",
            "permutation_rmse_increase_std",
            "impurity_importance",
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# 6. REDUNDÂNCIA ENTRE LAGS
# ------------------------------------------------------------
#
# Correlação entre preditores no TREINAMENTO.
# Não utiliza CO2 para calcular redundância.
# ------------------------------------------------------------

corr = X_train.corr(method="pearson")

redundancy_rows = []

for lag in range(HORIZON + 1):

    feature = FEATURES[lag]

    other_corr = corr.loc[
        feature
    ].drop(feature).abs()

    most_correlated = other_corr.idxmax()

    redundancy_rows.append({
        "lag_hours": lag,
        "feature": feature,
        "most_correlated_feature": most_correlated,
        "max_abs_correlation": float(
            other_corr.max()
        ),
        "mean_abs_correlation": float(
            other_corr.mean()
        ),
    })

redundancy_df = pd.DataFrame(redundancy_rows)

importance_df = importance_df.merge(
    redundancy_df,
    on=["feature", "lag_hours"],
    how="left",
    validate="one_to_one"
)

_project_print("\nREDUNDÂNCIA — TOP 20 LAGS")
_project_print(
    importance_df[
        [
            "lag_hours",
            "permutation_rmse_increase_mean",
            "max_abs_correlation",
            "most_correlated_feature",
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# 7. CANDIDATOS EXPLORATÓRIOS
# ------------------------------------------------------------
#
# Apenas ranking preliminar.
# A seleção efetiva será realizada na Célula 09.
# ------------------------------------------------------------

top_lags = (
    importance_df.loc[
        importance_df["lag_hours"] > 0
    ]
    .head(15)["lag_hours"]
    .astype(int)
    .tolist()
)

top_lags = sorted(top_lags)

_project_print("\n15 LAGS CANDIDATOS (EXPLORATÓRIOS)")
_project_print(top_lags)

# ------------------------------------------------------------
# 8. GRÁFICO DE IMPORTÂNCIA
# ------------------------------------------------------------

plot_df = importance_df.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(11, 8))

ax.barh(
    plot_df["feature"],
    plot_df["permutation_rmse_increase_mean"],
    xerr=plot_df["permutation_rmse_increase_std"],
)

ax.axvline(0, linestyle="--", linewidth=1)

ax.set_xlabel(
    "Aumento do RMSE após permutação (t CO2/h)"
)

ax.set_title(
    "Importância por permutação — RF, horizonte 72 h"
)

ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 9. MAPA DE CORRELAÇÕES ENTRE PREDITORES
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(
    corr.to_numpy(),
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
    aspect="auto",
)

ax.set_title(
    "Correlação entre G(t) e lags — treinamento"
)

ax.set_xlabel("Lag (horas)")
ax.set_ylabel("Lag (horas)")

ticks = [
    0, 12, 24, 36, 48, 60, 72
]

ax.set_xticks(ticks)
ax.set_yticks(ticks)

ax.set_xticklabels(ticks)
ax.set_yticklabels(ticks)

plt.colorbar(
    im,
    ax=ax,
    label="Correlação de Pearson"
)

plt.tight_layout()
plt.savefig(
    OUTPUT_HEATMAP,
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 10. EXPORTAÇÃO
# ------------------------------------------------------------

_project_csv(importance_df, 
    OUTPUT_IMPORTANCE,
    index=False
)

_project_csv(redundancy_df, 
    OUTPUT_REDUNDANCY,
    index=False
)

elapsed = time.time() - start_time

report = {
    "cell": "08/14",
    "project": "predictive_lag_selection",
    "source": str(INPUT_PATH),
    "horizon_hours": HORIZON,
    "train_rows": int(len(train)),
    "validation_rows": int(len(validation)),
    "features": FEATURES,
    "model": "random_forest",
    "random_forest_parameters": RF_PARAMS,
    "validation_rmse": rmse,
    "permutation_repeats": PERMUTATION_REPEATS,
    "importance_method": (
        "permutation_negative_rmse"
    ),
    "redundancy_method": (
        "pearson_predictor_correlation_train"
    ),
    "exploratory_top_15_lags": top_lags,
    "selection_final": False,
    "test_used": False,
    "execution_seconds": round(elapsed, 2),
    "outputs": {
        "importance": str(OUTPUT_IMPORTANCE),
        "redundancy": str(OUTPUT_REDUNDANCY),
        "importance_figure": str(OUTPUT_FIGURE),
        "redundancy_figure": str(OUTPUT_HEATMAP),
    },
}

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 11. INTEGRIDADE
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

for path in [
    OUTPUT_IMPORTANCE,
    OUTPUT_REDUNDANCY,
    OUTPUT_REPORT,
    OUTPUT_FIGURE,
    OUTPUT_HEATMAP,
]:
    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 08/14 CONCLUÍDA — "
    "IMPORTÂNCIA E REDUNDÂNCIA"
)
_project_print("=" * 72)

_project_print("Horizonte exploratório:", HORIZON, "h")
_project_print("Preditores avaliados:", len(FEATURES))
_project_print("RMSE de referência:", round(rmse, 4))
_project_print("Candidatos preliminares:", top_lags)
_project_print("Teste preservado.")
_project_print("Nenhum lag selecionado definitivamente.")
_project_print(
    "PRÓXIMA CÉLULA: 09 — "
    "seleção de subconjunto parcimonioso de lags."
)


In [ ]:
# Referência metodológica: Cawley e Talbot (2010). Ranking e escolha por desempenho de validação. Parcimônia e tolerância são regras do projeto.

# ============================================================
# CÉLULA 09/14 — SELEÇÃO PARCIMONIOSA DE LAGS
# ============================================================
#
# PROJETO: Seleção preditiva de lags para estimação de CO2
# UNIDADE: Iatan Unit 2 | ORISPL 6065 | 2025
#
# OBJETIVOS:
# 1. Comparar conjuntos reduzidos de lags.
# 2. Avaliar Ridge e Random Forest.
# 3. Utilizar uma amostra comum de treino e validação.
# 4. Selecionar um subconjunto preliminar parcimonioso.
# 5. Preservar o teste para a Célula 13.
#
# Não utiliza CO2 defasado ou variáveis de processos.
# ============================================================

from pathlib import Path
import json
import hashlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

IMPORTANCE_PATH = (
    REPORT_DIR / "cell08_lag_importance.csv"
)

CELL07_PATH = (
    REPORT_DIR / "cell07_horizon_comparison.csv"
)

OUTPUT_RESULTS = (
    REPORT_DIR / "cell09_parsimonious_lag_comparison.csv"
)

OUTPUT_SELECTION = (
    REPORT_DIR / "cell09_preliminary_lag_selection.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell09_parsimonious_lag_comparison.png"
)

OUTPUT_REPORT = (
    REPORT_DIR / "cell09_selection_audit.json"
)

TARGET = "co2_metric_tons"
CURRENT = "generation_mw"

REFERENCE_HORIZON = 72

RANDOM_STATE = 42
RIDGE_ALPHA = 1.0

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 3,
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

assert INPUT_PATH.exists()
assert IMPORTANCE_PATH.exists()
assert CELL07_PATH.exists()

# ------------------------------------------------------------
# 2. CARREGAMENTO
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_PATH)

importance = pd.read_csv(IMPORTANCE_PATH)
cell07 = pd.read_csv(CELL07_PATH)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="raise"
)

assert len(df) == 8760
assert df["timestamp"].is_unique
assert int(df["eligible_for_modeling"].sum()) == 6249

required_importance = {
    "lag_hours",
    "permutation_rmse_increase_mean",
}

assert required_importance.issubset(
    importance.columns
)

# ------------------------------------------------------------
# 3. DEFINIÇÃO DOS CANDIDATOS
# ------------------------------------------------------------
#
# Os conjuntos são construídos a partir do ranking
# de importância da Célula 08.
#
# G(t) está presente em todos os modelos.
#
# Os conjuntos Top-K são aninhados:
# Top-3 está contido em Top-5, e assim por diante.
# ------------------------------------------------------------

ranking = (
    importance.loc[
        (importance["lag_hours"] >= 1)
        & (importance["lag_hours"] <= REFERENCE_HORIZON)
    ]
    .sort_values(
        "permutation_rmse_increase_mean",
        ascending=False
    )["lag_hours"]
    .astype(int)
    .tolist()
)

assert len(ranking) == REFERENCE_HORIZON
assert len(set(ranking)) == REFERENCE_HORIZON

candidate_sets = {
    "baseline": [],
    "top_3": sorted(ranking[:3]),
    "top_5": sorted(ranking[:5]),
    "top_8": sorted(ranking[:8]),
    "top_10": sorted(ranking[:10]),
    "top_15": sorted(ranking[:15]),
    "full_72": list(range(1, 73)),
}

# ------------------------------------------------------------
# 4. AMOSTRA COMUM
# ------------------------------------------------------------
#
# Todos os modelos usam as mesmas observações que
# possuem G(t), CO2(t) e lags 1–72 completos.
#
# Essa restrição é usada somente para comparação.
# Não exclui definitivamente observações do projeto.
# ------------------------------------------------------------

full_features = [CURRENT] + [
    f"generation_lag_{k:03d}h"
    for k in range(1, 73)
]

required = [
    "timestamp",
    "partition",
    "eligible_for_modeling",
    TARGET,
] + full_features

missing = [
    c for c in required
    if c not in df.columns
]

assert not missing, f"Colunas ausentes: {missing}"

common_mask = (
    df["eligible_for_modeling"]
    & df[full_features + [TARGET]].notna().all(axis=1)
)

train = df.loc[
    common_mask & df["partition"].eq("train")
].copy()

validation = df.loc[
    common_mask & df["partition"].eq("validation")
].copy()

assert len(train) == 3025
assert len(validation) == 384

assert train["timestamp"].max() < (
    validation["timestamp"].min()
)

_project_print("Fonte:", INPUT_PATH)
_project_print("Treino comum:", len(train))
_project_print("Validação comum:", len(validation))
_project_print("Teste: reservado")
_project_print("\nRanking dos 15 primeiros lags:")
_project_print(ranking[:15])

_project_print("\nCONJUNTOS CANDIDATOS")
for name, lags in candidate_sets.items():
    _project_print(
        f"{name:12s} | "
        f"{len(lags):2d} lags | {lags}"
    )

# ------------------------------------------------------------
# 5. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def build_model(name):

    if name == "ridge":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=RIDGE_ALPHA)),
        ])

    if name == "random_forest":
        return RandomForestRegressor(
            **RF_PARAMS
        )

    raise ValueError(name)


def calculate_metrics(y_true, y_pred):

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
    }

# ------------------------------------------------------------
# 6. COMPARAÇÃO DOS CONJUNTOS
# ------------------------------------------------------------

results = []

start_time = time.time()

for candidate_name, lags in candidate_sets.items():

    features = [CURRENT] + [
        f"generation_lag_{k:03d}h"
        for k in lags
    ]

    for model_name in ["ridge", "random_forest"]:

        model = build_model(model_name)

        model.fit(
            train[features],
            train[TARGET]
        )

        predictions = model.predict(
            validation[features]
        )

        metrics = calculate_metrics(
            validation[TARGET],
            predictions
        )

        results.append({
            "candidate": candidate_name,
            "model": model_name,
            "n_lags": len(lags),
            "n_features": len(features),
            "lags": _project_dumps(lags),
            "train_rows": len(train),
            "validation_rows": len(validation),
            "rmse": metrics["rmse"],
            "mae": metrics["mae"],
            "r2": metrics["r2"],
        })

        _project_print(
            f"{candidate_name:12s} | "
            f"{model_name:13s} | "
            f"Lags={len(lags):2d} | "
            f"RMSE={metrics['rmse']:.4f}"
        )

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 7. GANHO SOBRE BASELINE COMUM
# ------------------------------------------------------------

baseline_by_model = (
    results_df.loc[
        results_df["candidate"] == "baseline"
    ]
    .set_index("model")["rmse"]
    .to_dict()
)

results_df["baseline_rmse_common"] = (
    results_df["model"].map(baseline_by_model)
)

results_df["rmse_gain_pct"] = (
    100
    * (
        results_df["baseline_rmse_common"]
        - results_df["rmse"]
    )
    / results_df["baseline_rmse_common"]
)

# ------------------------------------------------------------
# 8. AUDITORIA COM A CÉLULA 07
# ------------------------------------------------------------

for model_name in ["ridge", "random_forest"]:

    observed = results_df.loc[
        (results_df["candidate"] == "full_72")
        & (results_df["model"] == model_name),
        "rmse"
    ].iloc[0]

    reference = cell07.loc[
        (cell07["model"] == model_name)
        & (cell07["horizon_hours"] == 72),
        "horizon_rmse"
    ].iloc[0]

    assert np.isclose(
        observed,
        reference,
        rtol=1e-8,
        atol=1e-8
    ), (
        f"Divergência no modelo {model_name} "
        "em relação à Célula 07."
    )

_project_print("\nCOMPARAÇÃO CONSOLIDADA")
_project_print(
    results_df[
        [
            "candidate",
            "model",
            "n_lags",
            "rmse",
            "mae",
            "r2",
            "rmse_gain_pct",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 9. SELEÇÃO PRELIMINAR PARCIMONIOSA
# ------------------------------------------------------------
#
# Critério:
# 1. Identificar o menor RMSE do Random Forest.
# 2. Aceitar modelos com RMSE até 2% acima do melhor.
# 3. Entre eles, escolher o que usa menos lags.
# 4. Em caso de empate, escolher menor RMSE.
#
# Esta seleção ainda será avaliada nas Células 10–12.
# ------------------------------------------------------------

rf_results = results_df.loc[
    results_df["model"] == "random_forest"
].copy()

best_rmse = float(rf_results["rmse"].min())

TOLERANCE_PCT = 2.0

threshold = (
    best_rmse * (1 + TOLERANCE_PCT / 100)
)

acceptable = rf_results.loc[
    rf_results["rmse"] <= threshold
].copy()

acceptable = acceptable.sort_values(
    ["n_lags", "rmse", "candidate"]
)

selected = acceptable.iloc[0]

selected_lags = json.loads(
    selected["lags"]
)

_project_print("\nSELEÇÃO PRELIMINAR")
_project_print("Candidato:", selected["candidate"])
_project_print("Lags:", selected_lags)
_project_print("Número de lags:", len(selected_lags))
_project_print("RMSE:", round(selected["rmse"], 4))
_project_print("Melhor RMSE observado:", round(best_rmse, 4))
_project_print("Tolerância:", TOLERANCE_PCT, "%")

# ------------------------------------------------------------
# 10. GRÁFICO COMPARATIVO
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 6))

for model_name in ["ridge", "random_forest"]:

    subset = results_df.loc[
        results_df["model"] == model_name
    ].sort_values(["n_lags", "candidate"])

    ax.plot(
        subset["n_lags"],
        subset["rmse"],
        marker="o",
        label=model_name
    )

ax.set_xlabel("Número de lags")
ax.set_ylabel("RMSE (toneladas métricas/h)")
ax.set_title(
    "Seleção parcimoniosa — mesma amostra de validação"
)

ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 11. EXPORTAÇÃO
# ------------------------------------------------------------

_project_csv(results_df, 
    OUTPUT_RESULTS,
    index=False
)

selection = {
    "cell": "09/14",
    "selection_status": "preliminary",
    "model": "random_forest",
    "selected_candidate": str(
        selected["candidate"]
    ),
    "selected_lags_hours": selected_lags,
    "selected_n_lags": len(selected_lags),
    "selected_rmse": float(selected["rmse"]),
    "best_rmse": best_rmse,
    "tolerance_pct": TOLERANCE_PCT,
    "train_rows_common": int(len(train)),
    "validation_rows_common": int(len(validation)),
    "selection_rule": (
        "minimum_number_of_lags_within_2pct_"
        "of_best_validation_rmse"
    ),
    "test_used": False,
}

with open(
    OUTPUT_SELECTION,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        selection,
        f,
        ensure_ascii=False,
        indent=2
    )

elapsed = time.time() - start_time

report = {
    "cell": "09/14",
    "source": str(INPUT_PATH),
    "importance_source": str(IMPORTANCE_PATH),
    "reference_horizon": REFERENCE_HORIZON,
    "candidate_sets": candidate_sets,
    "models": ["ridge", "random_forest"],
    "train_rows_common": int(len(train)),
    "validation_rows_common": int(len(validation)),
    "selection": selection,
    "test_used": False,
    "execution_seconds": round(elapsed, 2),
    "results": results_df.to_dict(orient="records"),
}

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 12. INTEGRIDADE
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


for path in [
    OUTPUT_RESULTS,
    OUTPUT_SELECTION,
    OUTPUT_REPORT,
    OUTPUT_FIGURE,
]:

    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 09/14 CONCLUÍDA — "
    "SELEÇÃO PARCIMONIOSA"
)
_project_print("=" * 72)

_project_print("Conjuntos comparados:", len(candidate_sets))
_project_print("Modelos:", 2)
_project_print("Treino comum:", len(train))
_project_print("Validação comum:", len(validation))
_project_print("Lags preliminares:", selected_lags)
_project_print(f"Tempo de execução: {elapsed:.1f} s")
_project_print("Teste preservado.")
_project_print(
    "PRÓXIMA CÉLULA: 10 — "
    "avaliação de janelas móveis."
)


In [ ]:
# Referência metodológica: pandas.DataFrame.rolling. shift(1) exclui a hora corrente das médias do histórico.

# ============================================================
# CÉLULA 10/14 — AVALIAÇÃO DE JANELAS MÓVEIS DE GERAÇÃO
# ============================================================
#
# PROJETO:
# Seleção preditiva de lags para estimação horária de CO2
#
# UNIDADE:
# Iatan — Unit 2 — ORISPL 6065 — Ano 2025
#
# OBJETIVOS:
# 1. Construir médias móveis de geração.
# 2. Comparar o baseline, 8 lags e 72 lags.
# 3. Avaliar janelas isoladas e combinadas com 8 lags.
# 4. Identificar uma representação temporal compacta.
# 5. Preservar integralmente o teste.
#
# RESTRIÇÕES:
# - Somente geração elétrica como entrada.
# - Não utilizar CO2 defasado.
# - Não utilizar variáveis de processos.
# - Não utilizar informações futuras.
# - Não alterar os dados das células anteriores.
# ============================================================

from pathlib import Path
import json
import hashlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

CELL09_RESULTS = (
    REPORT_DIR / "cell09_parsimonious_lag_comparison.csv"
)

OUTPUT_RESULTS = (
    REPORT_DIR / "cell10_rolling_window_comparison.csv"
)

OUTPUT_FEATURES = (
    DATA_DIR / "cell10_rolling_window_features.parquet"
)

OUTPUT_SELECTION = (
    REPORT_DIR / "cell10_rolling_window_selection.json"
)

OUTPUT_AUDIT = (
    REPORT_DIR / "cell10_rolling_window_audit.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell10_rolling_window_comparison.png"
)

TARGET = "co2_metric_tons"
CURRENT = "generation_mw"

WINDOWS = [3, 6, 12, 24]

COMPACT_LAGS = [2, 20, 23, 24, 26, 29, 41, 52]

REFERENCE_HORIZON = 72

RIDGE_ALPHA = 1.0
RANDOM_STATE = 42

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 3,
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

assert INPUT_PATH.exists()
assert CELL09_RESULTS.exists()

# ------------------------------------------------------------
# 2. CARREGAMENTO E VERIFICAÇÕES
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_PATH)
cell09 = pd.read_csv(CELL09_RESULTS)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="raise"
)

df = df.sort_values("timestamp").reset_index(drop=True)

assert len(df) == 8760
assert df["timestamp"].is_unique
assert int(df["eligible_for_modeling"].sum()) == 6249

expected_timestamps = pd.date_range(
    start=df["timestamp"].min(),
    periods=len(df),
    freq="h"
)

assert (
    df["timestamp"].to_numpy()
    == expected_timestamps.to_numpy()
).all(), "A série não possui calendário horário contínuo."

_project_print("Fonte:", INPUT_PATH)
_project_print("Horas do calendário:", len(df))
_project_print("Janelas avaliadas:", WINDOWS)
_project_print("Lags compactos:", COMPACT_LAGS)
_project_print("Teste: reservado")

# ------------------------------------------------------------
# 3. CONSTRUÇÃO DAS MÉDIAS MÓVEIS
# ------------------------------------------------------------
#
# Definição:
#
# rolling_mean_003h(t)
#   = média de G(t-1), G(t-2), G(t-3)
#
# rolling_mean_006h(t)
#   = média de G(t-1), ..., G(t-6)
#
# E assim sucessivamente.
#
# A geração atual G(t) entra separadamente no modelo.
#
# As médias são calculadas no calendário completo,
# antes da seleção de treino e validação.
#
# min_periods = janela:
# não imputamos horas sem medição.
# ------------------------------------------------------------

rolling_features = []

past_generation = df[CURRENT].shift(1)

for window in WINDOWS:

    feature = f"generation_rolling_mean_{window:03d}h"

    df[feature] = (
        past_generation
        .rolling(
            window=window,
            min_periods=window
        )
        .mean()
    )

    rolling_features.append(feature)

# ------------------------------------------------------------
# 4. AUDITORIA DAS JANELAS
# ------------------------------------------------------------

for window in WINDOWS:

    feature = f"generation_rolling_mean_{window:03d}h"

    expected = (
        df[
            [
                f"generation_lag_{k:03d}h"
                for k in range(1, window + 1)
            ]
        ]
        .mean(
            axis=1,
            skipna=False
        )
    )

    assert np.allclose(
        df[feature].to_numpy(),
        expected.to_numpy(),
        equal_nan=True,
        rtol=1e-10,
        atol=1e-10
    ), f"Falha na auditoria da janela {window} h"

_project_print("\nAUDITORIA DAS MÉDIAS MÓVEIS")
for window in WINDOWS:

    feature = f"generation_rolling_mean_{window:03d}h"

    _project_print(
        f"Janela {window:2d} h | "
        f"Valores disponíveis: "
        f"{df[feature].notna().sum()}"
    )

# ------------------------------------------------------------
# 5. AMOSTRA COMUM DA CÉLULA 09
# ------------------------------------------------------------
#
# Mantemos o mesmo conjunto de linhas completas
# para G(t), CO2(t) e lags 1–72.
#
# Assim, diferenças de RMSE decorrem das variáveis
# utilizadas, não de mudanças na amostra.
# ------------------------------------------------------------

full_lag_features = [
    f"generation_lag_{k:03d}h"
    for k in range(1, REFERENCE_HORIZON + 1)
]

common_features = [CURRENT] + full_lag_features

common_mask = (
    df["eligible_for_modeling"]
    & df[common_features + [TARGET]]
        .notna()
        .all(axis=1)
)

train = df.loc[
    common_mask
    & df["partition"].eq("train")
].copy()

validation = df.loc[
    common_mask
    & df["partition"].eq("validation")
].copy()

assert len(train) == 3025
assert len(validation) == 384

assert (
    train[rolling_features].notna().all().all()
)

assert (
    validation[rolling_features].notna().all().all()
)

assert (
    train["timestamp"].max()
    < validation["timestamp"].min()
)

_project_print("\nAMOSTRA COMUM")
_project_print("Treino:", len(train))
_project_print("Validação:", len(validation))
_project_print("Teste: reservado")

# ------------------------------------------------------------
# 6. CONJUNTOS DE VARIÁVEIS
# ------------------------------------------------------------

compact_features = [
    f"generation_lag_{k:03d}h"
    for k in COMPACT_LAGS
]

feature_sets = {
    "baseline": [CURRENT],

    "rolling_3_6": [
        CURRENT,
        "generation_rolling_mean_003h",
        "generation_rolling_mean_006h",
    ],

    "rolling_3_6_12_24": (
        [CURRENT] + rolling_features
    ),

    "compact_8": (
        [CURRENT] + compact_features
    ),

    "compact_8_rolling_3_6": (
        [CURRENT]
        + compact_features
        + rolling_features[:2]
    ),

    "compact_8_rolling_all": (
        [CURRENT]
        + compact_features
        + rolling_features
    ),

    "full_72": (
        [CURRENT] + full_lag_features
    ),
}

assert len(feature_sets) == 7

for name, features in feature_sets.items():

    assert len(features) == len(set(features))

    assert set(features).issubset(df.columns)

_project_print("\nCONJUNTOS AVALIADOS")

for name, features in feature_sets.items():

    _project_print(
        f"{name:25s} | "
        f"{len(features):2d} preditores"
    )

# ------------------------------------------------------------
# 7. MODELOS E MÉTRICAS
# ------------------------------------------------------------

def build_model(model_name):

    if model_name == "ridge":

        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=RIDGE_ALPHA)),
        ])

    if model_name == "random_forest":

        return RandomForestRegressor(
            **RF_PARAMS
        )

    raise ValueError(model_name)


def calculate_metrics(y_true, y_pred):

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
    }

# ------------------------------------------------------------
# 8. EXPERIMENTO PREDITIVO
# ------------------------------------------------------------

results = []

start_time = time.time()

for candidate_name, features in feature_sets.items():

    for model_name in ["ridge", "random_forest"]:

        model = build_model(model_name)

        model.fit(
            train[features],
            train[TARGET]
        )

        predictions = model.predict(
            validation[features]
        )

        metrics = calculate_metrics(
            validation[TARGET],
            predictions
        )

        results.append({
            "candidate": candidate_name,
            "model": model_name,
            "n_features": len(features),
            "n_generation_lags": sum(
                "generation_lag_" in f
                for f in features
            ),
            "n_rolling_features": sum(
                "generation_rolling_mean_" in f
                for f in features
            ),
            "features": _project_dumps(features),
            "train_rows": len(train),
            "validation_rows": len(validation),
            **metrics,
        })

        _project_print(
            f"{candidate_name:25s} | "
            f"{model_name:13s} | "
            f"RMSE={metrics['rmse']:.4f}"
        )

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 9. GANHOS SOBRE O BASELINE
# ------------------------------------------------------------

baseline_by_model = (
    results_df.loc[
        results_df["candidate"] == "baseline"
    ]
    .set_index("model")["rmse"]
    .to_dict()
)

results_df["baseline_rmse_common"] = (
    results_df["model"].map(baseline_by_model)
)

results_df["rmse_gain_pct"] = (
    100
    * (
        results_df["baseline_rmse_common"]
        - results_df["rmse"]
    )
    / results_df["baseline_rmse_common"]
)

# ------------------------------------------------------------
# 10. VERIFICAÇÃO COM A CÉLULA 09
# ------------------------------------------------------------

reference_candidates = [
    "baseline",
    "compact_8",
    "full_72",
]

mapping = {
    "baseline": "baseline",
    "compact_8": "top_8",
    "full_72": "full_72",
}

for candidate in reference_candidates:

    for model_name in ["ridge", "random_forest"]:

        current_rmse = results_df.loc[
            (results_df["candidate"] == candidate)
            & (results_df["model"] == model_name),
            "rmse"
        ].iloc[0]

        previous_rmse = cell09.loc[
            (
                cell09["candidate"]
                == mapping[candidate]
            )
            & (cell09["model"] == model_name),
            "rmse"
        ].iloc[0]

        assert np.isclose(
            current_rmse,
            previous_rmse,
            rtol=1e-8,
            atol=1e-8
        ), (
            f"Divergência em {candidate} / {model_name}"
        )

_project_print("\nAUDITORIA COM A CÉLULA 09: OK")

# ------------------------------------------------------------
# 11. SELEÇÃO EXPLORATÓRIA
# ------------------------------------------------------------
#
# Selecionamos o menor RMSE entre os modelos
# Random Forest com até 13 preditores.
#
# Isso inclui:
# G(t) + 8 lags + até 4 médias móveis.
#
# O modelo completo com 73 preditores permanece
# como referência externa à seleção compacta.
#
# Esta decisão ainda não é definitiva.
# ------------------------------------------------------------

rf_results = results_df.loc[
    results_df["model"] == "random_forest"
].copy()

compact_candidates = rf_results.loc[
    rf_results["n_features"] <= 13
].copy()

compact_candidates = compact_candidates.sort_values(
    ["rmse", "n_features", "candidate"]
)

selected = compact_candidates.iloc[0]

selected_features = json.loads(
    selected["features"]
)

full_rmse = float(
    rf_results.loc[
        rf_results["candidate"] == "full_72",
        "rmse"
    ].iloc[0]
)

_project_print("\nRESULTADOS CONSOLIDADOS")
_project_print(
    results_df[
        [
            "candidate",
            "model",
            "n_features",
            "rmse",
            "mae",
            "r2",
            "rmse_gain_pct",
        ]
    ].to_string(index=False)
)

_project_print("\nMELHOR REPRESENTAÇÃO COMPACTA")
_project_print("Candidato:", selected["candidate"])
_project_print("Preditores:", len(selected_features))
_project_print("RMSE compacto:", round(selected["rmse"], 4))
_project_print("RMSE completo:", round(full_rmse, 4))
_project_print(
    "Diferença relativa ao completo:",
    round(
        100 * (
            selected["rmse"] / full_rmse - 1
        ),
        2
    ),
    "%"
)

# ------------------------------------------------------------
# 12. VISUALIZAÇÃO
# ------------------------------------------------------------

plot_df = results_df.pivot(
    index="candidate",
    columns="model",
    values="rmse"
)

plot_order = list(feature_sets.keys())

plot_df = plot_df.loc[plot_order]

ax = plot_df.plot(
    kind="bar",
    figsize=(13, 6),
    rot=35
)

ax.set_title(
    "Janelas móveis e lags — comparação na mesma validação"
)

ax.set_xlabel("Conjunto de preditores")
ax.set_ylabel("RMSE (toneladas métricas/h)")

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 13. EXPORTAÇÃO DAS FEATURES
# ------------------------------------------------------------
#
# Exportamos apenas:
# timestamp, partição, elegibilidade,
# G(t), CO2(t) e médias móveis.
#
# Não alteramos o banco original de lags.
# ------------------------------------------------------------

export_columns = [
    "timestamp",
    "partition",
    "eligible_for_modeling",
    CURRENT,
    TARGET,
] + rolling_features

df[export_columns].to_parquet(
    OUTPUT_FEATURES,
    index=False
)

_project_csv(results_df, 
    OUTPUT_RESULTS,
    index=False
)

selection = {
    "cell": "10/14",
    "selection_status": "exploratory",
    "selected_candidate": str(
        selected["candidate"]
    ),
    "selected_model": "random_forest",
    "selected_features": selected_features,
    "selected_n_features": len(selected_features),
    "selected_rmse": float(selected["rmse"]),
    "full_72_rmse": full_rmse,
    "train_rows_common": int(len(train)),
    "validation_rows_common": int(len(validation)),
    "test_used": False,
}

with open(
    OUTPUT_SELECTION,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        selection,
        f,
        ensure_ascii=False,
        indent=2
    )

elapsed = time.time() - start_time

audit = {
    "cell": "10/14",
    "source": str(INPUT_PATH),
    "rolling_windows_hours": WINDOWS,
    "rolling_definition": (
        "mean_of_generation_t_minus_1_to_t_minus_w"
    ),
    "rolling_requires_complete_window": True,
    "compact_lags_hours": COMPACT_LAGS,
    "candidate_sets": feature_sets,
    "train_rows_common": int(len(train)),
    "validation_rows_common": int(len(validation)),
    "selection": selection,
    "test_used": False,
    "execution_seconds": round(elapsed, 2),
    "results": results_df.to_dict(orient="records"),
}

with open(
    OUTPUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        audit,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 14. INTEGRIDADE
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


for path in [
    OUTPUT_RESULTS,
    OUTPUT_FEATURES,
    OUTPUT_SELECTION,
    OUTPUT_AUDIT,
    OUTPUT_FIGURE,
]:

    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

_project_print("\n" + "=" * 72)

_project_print(
    "CÉLULA 10/14 CONCLUÍDA — "
    "AVALIAÇÃO DE JANELAS MÓVEIS"
)

_project_print("=" * 72)

_project_print("Janelas:", WINDOWS)
_project_print("Conjuntos avaliados:", len(feature_sets))
_project_print("Treino comum:", len(train))
_project_print("Validação comum:", len(validation))
_project_print("Melhor compacto:", selected["candidate"])
_project_print("RMSE compacto:", round(selected["rmse"], 4))
_project_print("RMSE 72 lags:", round(full_rmse, 4))
_project_print(f"Tempo de execução: {elapsed:.1f} s")
_project_print("Teste preservado.")

_project_print(
    "PRÓXIMA CÉLULA: 11 — "
    "consolidação e comparação das representações."
)


In [ ]:
# Referência metodológica: Aula 08, pp. 31–32. Consolidação de cenários, métricas e critérios de decisão.

# ============================================================
# CÉLULA 11/14 — CONSOLIDAÇÃO DAS REPRESENTAÇÕES TEMPORAIS
# ============================================================
#
# PROJETO:
# Seleção preditiva de lags para estimação horária de CO2
#
# UNIDADE:
# Iatan — Unit 2 — ORISPL 6065 — Ano 2025
#
# OBJETIVOS:
# 1. Consolidar os resultados das Células 09 e 10.
# 2. Comparar desempenho e complexidade dos modelos.
# 3. Quantificar o ganho em relação ao baseline.
# 4. Identificar candidatos para robustez temporal.
# 5. Registrar as decisões metodológicas.
#
# IMPORTANTE:
# - Não realiza novos treinamentos.
# - Não acessa observações do conjunto de teste.
# - Não altera resultados das células anteriores.
# - Não utiliza resultados de experimentos anteriores.
# - A seleção final ocorrerá após a Célula 12.
# ============================================================

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CELL09_PATH = (
    REPORT_DIR / "cell09_parsimonious_lag_comparison.csv"
)

CELL10_PATH = (
    REPORT_DIR / "cell10_rolling_window_comparison.csv"
)

CELL10_SELECTION_PATH = (
    REPORT_DIR / "cell10_rolling_window_selection.json"
)

OUTPUT_COMPARISON = (
    REPORT_DIR / "cell11_consolidated_comparison.csv"
)

OUTPUT_CANDIDATES = (
    REPORT_DIR / "cell11_robustness_candidates.json"
)

OUTPUT_AUDIT = (
    REPORT_DIR / "cell11_consolidation_audit.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell11_performance_complexity.png"
)

for path in [
    CELL09_PATH,
    CELL10_PATH,
    CELL10_SELECTION_PATH,
]:
    assert path.exists(), f"Arquivo ausente: {path}"

# ------------------------------------------------------------
# 2. CARREGAMENTO
# ------------------------------------------------------------

cell09 = pd.read_csv(CELL09_PATH)
cell10 = pd.read_csv(CELL10_PATH)

with open(
    CELL10_SELECTION_PATH,
    "r",
    encoding="utf-8"
) as f:
    cell10_selection = json.load(f)

assert cell10_selection["test_used"] is False

# ------------------------------------------------------------
# 3. CANDIDATOS DA CONSOLIDAÇÃO
# ------------------------------------------------------------
#
# Os quatro modelos representam:
#
# baseline:
#   G(t)
#
# compact_8:
#   G(t) + 8 lags selecionados
#
# compact_8_rolling_3_6:
#   G(t) + 8 lags + médias móveis de 3 e 6 h
#
# full_72:
#   G(t) + lags consecutivos de 1 a 72 h
#
# Todos foram avaliados na mesma amostra.
# ------------------------------------------------------------

CANDIDATES = [
    "baseline",
    "compact_8",
    "compact_8_rolling_3_6",
    "full_72",
]

LABELS = {
    "baseline": "Baseline — G(t)",
    "compact_8": "Compacto — 8 lags",
    "compact_8_rolling_3_6": (
        "Compacto — 8 lags + médias 3/6 h"
    ),
    "full_72": "Completo — 72 lags",
}

MODELS = ["ridge", "random_forest"]

selected_rows = cell10.loc[
    cell10["candidate"].isin(CANDIDATES)
    & cell10["model"].isin(MODELS)
].copy()

assert len(selected_rows) == 8

assert not selected_rows.duplicated(
    ["candidate", "model"]
).any()

assert (
    selected_rows["train_rows"].nunique() == 1
)

assert (
    selected_rows["validation_rows"].nunique() == 1
)

TRAIN_ROWS = int(
    selected_rows["train_rows"].iloc[0]
)

VALIDATION_ROWS = int(
    selected_rows["validation_rows"].iloc[0]
)

assert TRAIN_ROWS == 3025
assert VALIDATION_ROWS == 384

# ------------------------------------------------------------
# 4. CONFERÊNCIA COM A CÉLULA 09
# ------------------------------------------------------------

mapping = {
    "baseline": "baseline",
    "compact_8": "top_8",
    "full_72": "full_72",
}

for current_name, previous_name in mapping.items():

    for model_name in MODELS:

        rmse_10 = selected_rows.loc[
            (selected_rows["candidate"] == current_name)
            & (selected_rows["model"] == model_name),
            "rmse"
        ].iloc[0]

        previous = cell09.loc[
            (cell09["candidate"] == previous_name)
            & (cell09["model"] == model_name)
        ]

        assert len(previous) == 1

        rmse_09 = previous["rmse"].iloc[0]

        assert np.isclose(
            rmse_10,
            rmse_09,
            rtol=1e-8,
            atol=1e-8
        ), (
            f"Divergência entre C09 e C10: "
            f"{current_name} / {model_name}"
        )

_project_print("Auditoria C09 × C10: OK")

# ------------------------------------------------------------
# 5. CONSOLIDAÇÃO DAS MÉTRICAS
# ------------------------------------------------------------

comparison = selected_rows[
    [
        "candidate",
        "model",
        "n_features",
        "n_generation_lags",
        "n_rolling_features",
        "train_rows",
        "validation_rows",
        "rmse",
        "mae",
        "r2",
    ]
].copy()

comparison["label"] = (
    comparison["candidate"].map(LABELS)
)

baseline_rmse = (
    comparison.loc[
        comparison["candidate"] == "baseline"
    ]
    .set_index("model")["rmse"]
    .to_dict()
)

comparison["baseline_rmse"] = (
    comparison["model"].map(baseline_rmse)
)

comparison["rmse_gain_vs_baseline_pct"] = (
    100
    * (
        comparison["baseline_rmse"]
        - comparison["rmse"]
    )
    / comparison["baseline_rmse"]
)

# ------------------------------------------------------------
# 6. REFERÊNCIA DO MODELO COMPLETO
# ------------------------------------------------------------

full_rmse_by_model = (
    comparison.loc[
        comparison["candidate"] == "full_72"
    ]
    .set_index("model")["rmse"]
    .to_dict()
)

comparison["full_72_rmse"] = (
    comparison["model"].map(full_rmse_by_model)
)

comparison["rmse_excess_vs_full_pct"] = (
    100
    * (
        comparison["rmse"]
        - comparison["full_72_rmse"]
    )
    / comparison["full_72_rmse"]
)

comparison["feature_reduction_vs_full_pct"] = (
    100
    * (
        73 - comparison["n_features"]
    )
    / 73
)

# ------------------------------------------------------------
# 7. FRONTEIRA DE PARETO
# ------------------------------------------------------------
#
# Um modelo é dominado se existir outro modelo:
#
# - com igual ou menor número de preditores;
# - com igual ou menor RMSE;
# - e com pelo menos uma vantagem estrita.
#
# Avaliação separada por família de modelo.
# ------------------------------------------------------------

comparison["pareto_efficient"] = False

for model_name in MODELS:

    subset = comparison.loc[
        comparison["model"] == model_name
    ]

    for idx, row in subset.iterrows():

        dominated = (
            (
                subset["n_features"]
                <= row["n_features"]
            )
            &
            (
                subset["rmse"]
                <= row["rmse"]
            )
            &
            (
                (subset["n_features"] < row["n_features"])
                |
                (subset["rmse"] < row["rmse"])
            )
        ).any()

        comparison.loc[
            idx,
            "pareto_efficient"
        ] = not bool(dominated)

# ------------------------------------------------------------
# 8. CANDIDATOS PARA ROBUSTEZ
# ------------------------------------------------------------
#
# Decisão metodológica:
#
# Não escolher definitivamente um modelo apenas
# pelo RMSE observado em uma validação.
#
# Os quatro modelos RF seguem para robustez:
#
# 1. Baseline: referência obrigatória.
# 2. Compacto 8: menor complexidade.
# 3. Compacto 8 + médias: melhor compacto observado.
# 4. Completo 72: melhor RMSE observado.
#
# A Célula 12 avaliará a estabilidade temporal.
# ------------------------------------------------------------

robustness_candidates = [
    "baseline",
    "compact_8",
    "compact_8_rolling_3_6",
    "full_72",
]

rf_comparison = comparison.loc[
    comparison["model"] == "random_forest"
].copy()

rf_comparison = rf_comparison.set_index(
    "candidate"
).loc[robustness_candidates].reset_index()

_project_print("\n" + "=" * 72)
_project_print("CONSOLIDAÇÃO — RANDOM FOREST")
_project_print("=" * 72)

_project_print(
    rf_comparison[
        [
            "candidate",
            "n_features",
            "rmse",
            "mae",
            "r2",
            "rmse_gain_vs_baseline_pct",
            "rmse_excess_vs_full_pct",
            "feature_reduction_vs_full_pct",
            "pareto_efficient",
        ]
    ].round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 9. GRÁFICO DE DESEMPENHO × COMPLEXIDADE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for model_name, marker in [
    ("ridge", "s"),
    ("random_forest", "o"),
]:

    subset = comparison.loc[
        comparison["model"] == model_name
    ].sort_values("n_features")

    ax.plot(
        subset["n_features"],
        subset["rmse"],
        marker=marker,
        linewidth=1.8,
        label=model_name
    )

    for _, row in subset.iterrows():

        ax.annotate(
            row["candidate"],
            (
                row["n_features"],
                row["rmse"]
            ),
            xytext=(5, 6),
            textcoords="offset points",
            fontsize=8
        )

ax.set_xlabel("Número de preditores")
ax.set_ylabel("RMSE (toneladas métricas de CO2/h)")

ax.set_title(
    "Compromisso entre desempenho preditivo "
    "e complexidade"
)

ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 10. EXPORTAÇÃO DOS RESULTADOS
# ------------------------------------------------------------

comparison = comparison.sort_values(
    ["model", "n_features", "candidate"]
).reset_index(drop=True)

_project_csv(comparison, 
    OUTPUT_COMPARISON,
    index=False
)

COMPACT_LAGS = [2, 20, 23, 24, 26, 29, 41, 52]

feature_sets = {
    "baseline": [
        "generation_mw"
    ],
    "compact_8": (
        ["generation_mw"]
        + [
            f"generation_lag_{k:03d}h"
            for k in COMPACT_LAGS
        ]
    ),
    "compact_8_rolling_3_6": (
        ["generation_mw"]
        + [
            f"generation_lag_{k:03d}h"
            for k in COMPACT_LAGS
        ]
        + [
            "generation_rolling_mean_003h",
            "generation_rolling_mean_006h",
        ]
    ),
    "full_72": (
        ["generation_mw"]
        + [
            f"generation_lag_{k:03d}h"
            for k in range(1, 73)
        ]
    ),
}

# Conferir definições contra a Célula 10.
for candidate, expected_features in feature_sets.items():

    previous = cell10.loc[
        (cell10["candidate"] == candidate)
        & (cell10["model"] == "random_forest")
    ]

    assert len(previous) == 1

    stored_features = json.loads(
        previous["features"].iloc[0]
    )

    assert stored_features == expected_features

candidate_config = {
    "cell": "11/14",
    "status": "candidates_for_temporal_robustness",
    "unit": "Iatan Unit 2",
    "orispl_code": 6065,
    "year": 2025,
    "target": "co2_metric_tons",
    "model_family": "random_forest",
    "candidate_names": robustness_candidates,
    "feature_sets": feature_sets,
    "train_rows_previous_common": TRAIN_ROWS,
    "validation_rows_previous_common": VALIDATION_ROWS,
    "random_forest_parameters": {
        "n_estimators": 300,
        "max_depth": 12,
        "min_samples_leaf": 3,
        "max_features": 1.0,
        "random_state": 42,
        "n_jobs": -1,
    },
    "rolling_definition": (
        "mean_of_generation_t_minus_1_to_t_minus_w"
    ),
    "test_used": False,
    "final_model_selected": False,
}

with open(
    OUTPUT_CANDIDATES,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        candidate_config,
        f,
        ensure_ascii=False,
        indent=2
    )

audit = {
    "cell": "11/14",
    "sources": {
        "cell09": str(CELL09_PATH),
        "cell10": str(CELL10_PATH),
        "cell10_selection": str(
            CELL10_SELECTION_PATH
        ),
    },
    "comparison_rows": int(len(comparison)),
    "train_rows_common": TRAIN_ROWS,
    "validation_rows_common": VALIDATION_ROWS,
    "candidate_names": robustness_candidates,
    "models_compared": MODELS,
    "selection_status": (
        "pending_temporal_robustness"
    ),
    "test_used": False,
    "final_model_selected": False,
}

with open(
    OUTPUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 11. INTEGRIDADE
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


for path in [
    OUTPUT_COMPARISON,
    OUTPUT_CANDIDATES,
    OUTPUT_AUDIT,
    OUTPUT_FIGURE,
]:

    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

# ------------------------------------------------------------
# 12. RESUMO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 72)
_project_print(
    "CÉLULA 11/14 CONCLUÍDA — "
    "CONSOLIDAÇÃO DAS REPRESENTAÇÕES"
)
_project_print("=" * 72)

_project_print("Modelos consolidados:", len(comparison))
_project_print("Candidatos para robustez:", len(robustness_candidates))
_project_print("Treino comum anterior:", TRAIN_ROWS)
_project_print("Validação comum anterior:", VALIDATION_ROWS)

_project_print("\nCandidatos:")
for name in robustness_candidates:
    _project_print(
        f"- {name}: "
        f"{len(feature_sets[name])} preditores"
    )

_project_print("\nTeste preservado.")
_project_print("Nenhum modelo selecionado definitivamente.")

_project_print(
    "PRÓXIMA CÉLULA: 12 — "
    "análise de robustez temporal."
)


In [ ]:
# Referência metodológica: Cerqueira et al. (2020) e Cawley e Talbot (2010). Janelas cronológicas e seleção antes da avaliação.

# ============================================================
# CÉLULA 12/14 — ROBUSTEZ TEMPORAL
# ============================================================
#
# PROJETO:
# Seleção preditiva de lags para estimação horária de CO2
#
# UNIDADE:
# Iatan — Unit 2 — ORISPL 6065 — 2025
#
# OBJETIVOS:
# 1. Avaliar estabilidade temporal dos quatro candidatos.
# 2. Utilizar validação cronológica progressiva.
# 3. Comparar os modelos nas mesmas observações.
# 4. Medir a estabilidade dos ganhos sobre o baseline.
# 5. Congelar candidatos para o teste retrospectivo.
#
# RESTRIÇÕES:
# - Somente geração elétrica como entrada.
# - Sem emissões defasadas.
# - Sem HMM ou Process Mining.
# - Sem imputação automática de dados ausentes.
# - Sem acesso ao conjunto de teste.
# - Hiperparâmetros fixos.
#
# IMPORTANTE:
# Esta célula não utiliza as métricas do teste.
# ============================================================

from pathlib import Path
import json
import hashlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_LAGS = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

INPUT_ROLLING = (
    DATA_DIR / "cell10_rolling_window_features.parquet"
)

INPUT_CANDIDATES = (
    REPORT_DIR / "cell11_robustness_candidates.json"
)

OUTPUT_FOLDS = (
    REPORT_DIR / "cell12_temporal_robustness_folds.csv"
)

OUTPUT_SUMMARY = (
    REPORT_DIR / "cell12_temporal_robustness_summary.csv"
)

OUTPUT_DECISION = (
    REPORT_DIR / "cell12_pretest_model_decision.json"
)

OUTPUT_AUDIT = (
    REPORT_DIR / "cell12_temporal_robustness_audit.json"
)

OUTPUT_FIGURE = (
    FIG_DIR / "cell12_temporal_robustness.png"
)

for path in [
    INPUT_LAGS,
    INPUT_ROLLING,
    INPUT_CANDIDATES,
]:
    assert path.exists(), f"Arquivo ausente: {path}"

# ------------------------------------------------------------
# 2. CARREGAMENTO DAS CONFIGURAÇÕES CONGELADAS
# ------------------------------------------------------------

with open(
    INPUT_CANDIDATES,
    "r",
    encoding="utf-8"
) as f:
    config = json.load(f)

assert config["test_used"] is False
assert config["final_model_selected"] is False

candidate_names = config["candidate_names"]
feature_sets = config["feature_sets"]
rf_params = config["random_forest_parameters"]

expected_candidates = [
    "baseline",
    "compact_8",
    "compact_8_rolling_3_6",
    "full_72",
]

assert candidate_names == expected_candidates

# ------------------------------------------------------------
# 3. CARREGAMENTO DOS DADOS
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_LAGS)

rolling = pd.read_parquet(INPUT_ROLLING)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="raise"
)

rolling["timestamp"] = pd.to_datetime(
    rolling["timestamp"],
    errors="raise"
)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

rolling = rolling.sort_values(
    "timestamp"
).reset_index(drop=True)

assert len(df) == 8760
assert len(rolling) == 8760

assert df["timestamp"].is_unique
assert rolling["timestamp"].is_unique

rolling_columns = [
    "generation_rolling_mean_003h",
    "generation_rolling_mean_006h",
]

df = df.merge(
    rolling[["timestamp"] + rolling_columns],
    on="timestamp",
    how="left",
    validate="one_to_one"
)

assert len(df) == 8760

# ------------------------------------------------------------
# 4. PROTEÇÃO DO CONJUNTO DE TESTE
# ------------------------------------------------------------
#
# O filtro ocorre antes de qualquer treinamento
# ou avaliação de métricas.
#
# Não utilizamos CO2 de teste nem observações de
# teste para selecionar os modelos.
# ------------------------------------------------------------

development = df.loc[
    df["partition"].isin(
        ["train", "validation"]
    )
].copy()

assert not development["partition"].eq("test").any()

development = development.sort_values(
    "timestamp"
).reset_index(drop=True)

_project_print("Horas do calendário:", len(df))
_project_print("Horas no desenvolvimento:", len(development))
_project_print("Teste: reservado")

# ------------------------------------------------------------
# 5. AMOSTRA COMUM
# ------------------------------------------------------------
#
# Para garantir comparabilidade, exigimos:
# - G(t) observado;
# - CO2(t) observado;
# - lags 1–72 disponíveis.
#
# Essa é a mesma lógica das Células 09–11.
#
# As observações permanecem em ordem temporal.
# ------------------------------------------------------------

TARGET = "co2_metric_tons"

all_features = feature_sets["full_72"]

common_mask = (
    development["eligible_for_modeling"]
    & development[all_features + [TARGET]]
        .notna()
        .all(axis=1)
)

common = development.loc[
    common_mask
].copy()

common = common.sort_values(
    "timestamp"
).reset_index(drop=True)

assert len(common) > 1000

for candidate in candidate_names:

    features = feature_sets[candidate]

    assert common[features].notna().all().all()

_project_print("Observações comuns:", len(common))
_project_print("Primeira observação:", common["timestamp"].min())
_project_print("Última observação:", common["timestamp"].max())

# ------------------------------------------------------------
# 6. DIVISÕES TEMPORAIS PROGRESSIVAS
# ------------------------------------------------------------
#
# Utilizamos 4 folds de validação.
#
# O primeiro treinamento corresponde a 50%
# das observações comuns de desenvolvimento.
#
# Os 50% restantes são divididos em 4 blocos
# consecutivos de validação.
#
# O treinamento cresce progressivamente:
#
# Fold 1: primeiros 50% -> bloco seguinte
# Fold 2: primeiros 62,5% -> bloco seguinte
# Fold 3: primeiros 75% -> bloco seguinte
# Fold 4: primeiros 87,5% -> bloco final
#
# A ordem cronológica é preservada.
# ------------------------------------------------------------

N_FOLDS = 4

n = len(common)

initial_train_size = int(
    np.floor(0.50 * n)
)

remaining = n - initial_train_size

validation_blocks = np.array_split(
    np.arange(initial_train_size, n),
    N_FOLDS
)

folds = []

for fold_number, val_idx in enumerate(
    validation_blocks,
    start=1
):

    train_end = int(val_idx[0])

    train_idx = np.arange(0, train_end)

    assert train_idx[-1] < val_idx[0]

    train_df = common.iloc[train_idx]
    val_df = common.iloc[val_idx]

    assert (
        train_df["timestamp"].max()
        < val_df["timestamp"].min()
    )

    folds.append({
        "fold": fold_number,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "train_start": str(
            train_df["timestamp"].min()
        ),
        "train_end": str(
            train_df["timestamp"].max()
        ),
        "val_start": str(
            val_df["timestamp"].min()
        ),
        "val_end": str(
            val_df["timestamp"].max()
        ),
    })

_project_print("\nDIVISÕES TEMPORAIS")

for fold in folds:

    _project_print(
        f"Fold {fold['fold']} | "
        f"Treino={len(fold['train_idx'])} | "
        f"Validação={len(fold['val_idx'])} | "
        f"{fold['val_start']} -> {fold['val_end']}"
    )

# ------------------------------------------------------------
# 7. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def calculate_metrics(y_true, y_pred):

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
    }


def make_model():

    return RandomForestRegressor(
        **rf_params
    )

# ------------------------------------------------------------
# 8. TREINAMENTO E AVALIAÇÃO POR FOLD
# ------------------------------------------------------------

results = []

start_time = time.time()

for fold in folds:

    fold_number = fold["fold"]

    train_df = common.iloc[
        fold["train_idx"]
    ]

    val_df = common.iloc[
        fold["val_idx"]
    ]

    for candidate in candidate_names:

        features = feature_sets[candidate]

        model = make_model()

        model.fit(
            train_df[features],
            train_df[TARGET]
        )

        predictions = model.predict(
            val_df[features]
        )

        metrics = calculate_metrics(
            val_df[TARGET],
            predictions
        )

        results.append({
            "fold": fold_number,
            "candidate": candidate,
            "n_features": len(features),
            "train_rows": len(train_df),
            "validation_rows": len(val_df),
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "validation_start": fold["val_start"],
            "validation_end": fold["val_end"],
            **metrics,
        })

        _project_print(
            f"Fold={fold_number} | "
            f"{candidate:25s} | "
            f"RMSE={metrics['rmse']:.4f} | "
            f"MAE={metrics['mae']:.4f}"
        )

fold_results = pd.DataFrame(results)

assert len(fold_results) == (
    N_FOLDS * len(candidate_names)
)

# ------------------------------------------------------------
# 9. GANHOS PAREADOS SOBRE BASELINE
# ------------------------------------------------------------

baseline = (
    fold_results.loc[
        fold_results["candidate"] == "baseline",
        ["fold", "rmse"]
    ]
    .rename(
        columns={"rmse": "baseline_rmse"}
    )
)

fold_results = fold_results.merge(
    baseline,
    on="fold",
    how="left",
    validate="many_to_one"
)

fold_results["rmse_gain_pct"] = (
    100
    * (
        fold_results["baseline_rmse"]
        - fold_results["rmse"]
    )
    / fold_results["baseline_rmse"]
)

# ------------------------------------------------------------
# 10. RESUMO DE ROBUSTEZ
# ------------------------------------------------------------

summary = (
    fold_results.groupby(
        "candidate",
        as_index=False
    )
    .agg(
        n_features=("n_features", "first"),
        folds=("fold", "count"),
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        rmse_min=("rmse", "min"),
        rmse_max=("rmse", "max"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        gain_mean_pct=("rmse_gain_pct", "mean"),
        gain_min_pct=("rmse_gain_pct", "min"),
        gain_max_pct=("rmse_gain_pct", "max"),
    )
)

summary["rmse_cv_pct"] = (
    100
    * summary["rmse_std"]
    / summary["rmse_mean"]
)

summary = summary.sort_values(
    ["rmse_mean", "n_features"]
).reset_index(drop=True)

_project_print("\n" + "=" * 72)
_project_print("RESUMO DA ROBUSTEZ TEMPORAL")
_project_print("=" * 72)

_project_print(
    summary.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 11. DECISÃO PRÉ-TESTE
# ------------------------------------------------------------
#
# Critério de decisão:
#
# - O modelo com menor RMSE médio é a referência.
# - Modelos até 5% acima desse RMSE são considerados
#   competitivos para a etapa de teste.
# - Entre os competitivos, priorizamos menos variáveis.
#
# Essa regra é aplicada somente aos folds de
# desenvolvimento, nunca ao teste.
#
# O baseline permanece como referência obrigatória.
# ------------------------------------------------------------

TOLERANCE_PCT = 5.0

nonbaseline = summary.loc[
    summary["candidate"] != "baseline"
].copy()

best_rmse = float(
    nonbaseline["rmse_mean"].min()
)

threshold = (
    best_rmse
    * (1 + TOLERANCE_PCT / 100)
)

competitive = nonbaseline.loc[
    nonbaseline["rmse_mean"] <= threshold
].copy()

competitive = competitive.sort_values(
    ["n_features", "rmse_mean", "candidate"]
)

selected = competitive.iloc[0]

selected_candidate = str(
    selected["candidate"]
)

_project_print("\nDECISÃO PRÉ-TESTE")
_project_print("Melhor RMSE médio:", round(best_rmse, 4))
_project_print("Tolerância:", TOLERANCE_PCT, "%")
_project_print("Candidato selecionado:", selected_candidate)
_project_print("Preditores:", int(selected["n_features"]))
_project_print(
    "RMSE médio:",
    round(selected["rmse_mean"], 4)
)

# ------------------------------------------------------------
# 12. GRÁFICO DE ROBUSTEZ
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 6))

for candidate in candidate_names:

    subset = fold_results.loc[
        fold_results["candidate"] == candidate
    ].sort_values("fold")

    ax.plot(
        subset["fold"],
        subset["rmse"],
        marker="o",
        linewidth=1.8,
        label=candidate
    )

ax.set_xlabel("Fold temporal")
ax.set_ylabel("RMSE (toneladas métricas de CO2/h)")

ax.set_title(
    "Robustez temporal — validação progressiva"
)

ax.set_xticks(range(1, N_FOLDS + 1))

ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 13. EXPORTAÇÃO
# ------------------------------------------------------------

_project_csv(fold_results, 
    OUTPUT_FOLDS,
    index=False
)

_project_csv(summary, 
    OUTPUT_SUMMARY,
    index=False
)

decision = {
    "cell": "12/14",
    "selection_status": "pretest_frozen",
    "selected_candidate": selected_candidate,
    "selected_features": feature_sets[
        selected_candidate
    ],
    "selected_n_features": int(
        selected["n_features"]
    ),
    "selected_rmse_mean": float(
        selected["rmse_mean"]
    ),
    "best_rmse_mean": best_rmse,
    "tolerance_pct": TOLERANCE_PCT,
    "competitive_candidates": (
        competitive["candidate"].tolist()
    ),
    "baseline_candidate": "baseline",
    "reference_full_candidate": "full_72",
    "model_family": "random_forest",
    "model_parameters": rf_params,
    "test_used": False,
}

with open(
    OUTPUT_DECISION,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        decision,
        f,
        ensure_ascii=False,
        indent=2
    )

elapsed = time.time() - start_time

audit = {
    "cell": "12/14",
    "method": "expanding_window_validation",
    "n_folds": N_FOLDS,
    "initial_training_fraction": 0.50,
    "common_rows": int(len(common)),
    "candidate_names": candidate_names,
    "feature_sets": feature_sets,
    "folds": [
        {
            key: value
            for key, value in fold.items()
            if key not in ["train_idx", "val_idx"]
        }
        for fold in folds
    ],
    "decision": decision,
    "test_used": False,
    "execution_seconds": round(elapsed, 2),
}

with open(
    OUTPUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        audit,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 14. INTEGRIDADE
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


for path in [
    OUTPUT_FOLDS,
    OUTPUT_SUMMARY,
    OUTPUT_DECISION,
    OUTPUT_AUDIT,
    OUTPUT_FIGURE,
]:

    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

# ------------------------------------------------------------
# 15. RESUMO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 72)

_project_print(
    "CÉLULA 12/14 CONCLUÍDA — "
    "ROBUSTEZ TEMPORAL"
)

_project_print("=" * 72)

_project_print("Folds:", N_FOLDS)
_project_print("Modelos:", len(candidate_names))
_project_print("Treinamentos:", len(fold_results))
_project_print("Candidato pré-teste:", selected_candidate)
_project_print(f"Tempo de execução: {elapsed:.1f} s")
_project_print("Teste preservado.")

_project_print(
    "PRÓXIMA CÉLULA: 13 — "
    "avaliação final retrospectiva."
)


In [ ]:
# Referência metodológica: Hyndman e Koehler (2006). RMSE e MAE na mesma amostra. A data já consultada torna a avaliação retrospectiva.

# ============================================================
# CÉLULA 13/14 — AVALIAÇÃO FINAL RETROSPECTIVA
# ============================================================
#
# PROJETO:
# Seleção preditiva de lags para estimação horária de CO2
#
# UNIDADE:
# Iatan — Unit 2 — ORISPL 6065 — Ano 2025
#
# OBJETIVOS:
# 1. Recuperar a decisão pré-teste da Célula 12.
# 2. Reajustar os quatro modelos congelados.
# 3. Avaliar no mesmo período de teste do estudo original.
# 4. Comparar os modelos nas mesmas observações.
# 5. Exportar métricas, previsões e gráficos.
#
# RESTRIÇÕES:
# - Apenas geração elétrica como entrada.
# - Sem CO2 defasado.
# - Sem HMM ou Process Mining.
# - Sem imputação automática de valores ausentes.
# - Sem ajuste de hiperparâmetros usando o teste.
# - Sem nova seleção de lags após observar o teste.
#
# IMPORTANTE:
# Esta célula abre o teste pela primeira vez.
# A decisão pré-teste permanece congelada.
# ============================================================

from pathlib import Path
import json
import hashlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


DATA_DIR = RUN_DIR / "data/processed"
REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_LAGS = (
    DATA_DIR / "cell04_generation_lag_bank_168h.parquet"
)

INPUT_ROLLING = (
    DATA_DIR / "cell10_rolling_window_features.parquet"
)

INPUT_CANDIDATES = (
    REPORT_DIR / "cell11_robustness_candidates.json"
)

INPUT_DECISION = (
    REPORT_DIR / "cell12_pretest_model_decision.json"
)

OUTPUT_METRICS = (
    REPORT_DIR / "cell13_final_test_metrics.csv"
)

OUTPUT_PREDICTIONS = (
    DATA_DIR / "cell13_final_test_predictions.parquet"
)

OUTPUT_AUDIT = (
    REPORT_DIR / "cell13_final_test_audit.json"
)

OUTPUT_FIGURE_SERIES = (
    FIG_DIR / "cell13_observed_vs_predicted.png"
)

OUTPUT_FIGURE_COMPARISON = (
    FIG_DIR / "cell13_final_test_comparison.png"
)

for path in [
    INPUT_LAGS,
    INPUT_ROLLING,
    INPUT_CANDIDATES,
    INPUT_DECISION,
]:
    assert path.exists(), f"Arquivo ausente: {path}"

# ------------------------------------------------------------
# 2. RECUPERAÇÃO DA DECISÃO PRÉ-TESTE
# ------------------------------------------------------------

with open(
    INPUT_CANDIDATES,
    "r",
    encoding="utf-8"
) as f:
    candidates_config = json.load(f)

with open(
    INPUT_DECISION,
    "r",
    encoding="utf-8"
) as f:
    pretest_decision = json.load(f)

assert candidates_config["test_used"] is False
assert pretest_decision["test_used"] is False

assert (
    pretest_decision["selection_status"]
    == "pretest_frozen"
)

candidate_names = candidates_config[
    "candidate_names"
]

feature_sets = candidates_config[
    "feature_sets"
]

rf_params = pretest_decision[
    "model_parameters"
]

selected_candidate = pretest_decision[
    "selected_candidate"
]

assert selected_candidate == "compact_8"

assert candidate_names == [
    "baseline",
    "compact_8",
    "compact_8_rolling_3_6",
    "full_72",
]

assert (
    pretest_decision["selected_features"]
    == feature_sets[selected_candidate]
)

assert (
    pretest_decision["model_family"]
    == "random_forest"
)

assert (
    rf_params
    == candidates_config[
        "random_forest_parameters"
    ]
)

_project_print("=" * 72)
_project_print("DECISÃO CONGELADA ANTES DO TESTE")
_project_print("=" * 72)

_project_print("Modelo prioritário:", selected_candidate)
_project_print(
    "Número de preditores:",
    len(feature_sets[selected_candidate])
)
_project_print(
    "Variáveis:",
    feature_sets[selected_candidate]
)
_project_print("Hiperparâmetros:", rf_params)

# ------------------------------------------------------------
# 3. CARREGAMENTO DOS DADOS
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_LAGS)

rolling = pd.read_parquet(INPUT_ROLLING)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="raise"
)

rolling["timestamp"] = pd.to_datetime(
    rolling["timestamp"],
    errors="raise"
)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

rolling = rolling.sort_values(
    "timestamp"
).reset_index(drop=True)

assert len(df) == 8760
assert len(rolling) == 8760

assert df["timestamp"].is_unique
assert rolling["timestamp"].is_unique

rolling_columns = [
    "generation_rolling_mean_003h",
    "generation_rolling_mean_006h",
]

df = df.merge(
    rolling[["timestamp"] + rolling_columns],
    on="timestamp",
    how="left",
    validate="one_to_one"
)

assert len(df) == 8760

# ------------------------------------------------------------
# 4. AUDITORIA DAS PARTIÇÕES
# ------------------------------------------------------------

TARGET = "co2_metric_tons"

development = df.loc[
    df["partition"].isin(
        ["train", "validation"]
    )
].copy()

test = df.loc[
    df["partition"].eq("test")
].copy()

assert len(development) > 0
assert len(test) > 0

assert (
    development["timestamp"].max()
    < test["timestamp"].min()
)

assert not (
    development["timestamp"].isin(
        test["timestamp"]
    )
).any()

_project_print("\nPARTIÇÕES")
_project_print("Horas de desenvolvimento:", len(development))
_project_print("Horas de teste:", len(test))
_project_print(
    "Início do teste:",
    test["timestamp"].min()
)
_project_print(
    "Fim do teste:",
    test["timestamp"].max()
)

# ------------------------------------------------------------
# 5. AMOSTRA COMUM PARA COMPARAÇÃO
# ------------------------------------------------------------
#
# Exigimos G(t), CO2(t) e lags 1–72 completos.
#
# Isso permite comparar todos os modelos nas mesmas
# observações de desenvolvimento e teste.
#
# A amostra é definida pela disponibilidade dos
# dados, nunca pelo valor observado de CO2.
#
# O histórico de geração pode atravessar a fronteira
# entre desenvolvimento e teste, pois utiliza apenas
# valores anteriores ao instante de previsão.
# ------------------------------------------------------------

full_features = feature_sets["full_72"]

required_columns = (
    full_features + [TARGET]
)

development_mask = (
    development["eligible_for_modeling"]
    & development[required_columns]
        .notna()
        .all(axis=1)
)

test_mask = (
    test["eligible_for_modeling"]
    & test[required_columns]
        .notna()
        .all(axis=1)
)

development_common = development.loc[
    development_mask
].copy()

test_common = test.loc[
    test_mask
].copy()

development_common = development_common.sort_values(
    "timestamp"
).reset_index(drop=True)

test_common = test_common.sort_values(
    "timestamp"
).reset_index(drop=True)

assert len(development_common) == 3409

assert len(test_common) > 0

assert (
    development_common["timestamp"].max()
    < test_common["timestamp"].min()
)

for candidate in candidate_names:

    features = feature_sets[candidate]

    assert (
        development_common[features]
        .notna()
        .all()
        .all()
    )

    assert (
        test_common[features]
        .notna()
        .all()
        .all()
    )

_project_print("\nAMOSTRA COMUM FINAL")
_project_print(
    "Desenvolvimento:",
    len(development_common)
)
_project_print("Teste:", len(test_common))

_project_print(
    "Primeiro teste elegível:",
    test_common["timestamp"].min()
)
_project_print(
    "Último teste elegível:",
    test_common["timestamp"].max()
)

# ------------------------------------------------------------
# 6. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def calculate_metrics(y_true, y_pred):

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
        "bias": float(
            np.mean(y_pred - y_true)
        ),
    }


def make_model():

    return RandomForestRegressor(
        **rf_params
    )

# ------------------------------------------------------------
# 7. TREINAMENTO FINAL E TESTE
# ------------------------------------------------------------
#
# Todos os modelos são reajustados exclusivamente
# nos dados de desenvolvimento.
#
# As previsões são realizadas uma única vez.
#
# Nenhum parâmetro será alterado com base
# nas métricas obtidas.
# ------------------------------------------------------------

results = []

predictions_df = test_common[
    ["timestamp", TARGET]
].copy()

predictions_df = predictions_df.rename(
    columns={
        TARGET: "co2_observed_metric_tons"
    }
)

start_time = time.time()

for candidate in candidate_names:

    features = feature_sets[candidate]

    model = make_model()

    model.fit(
        development_common[features],
        development_common[TARGET]
    )

    predictions = model.predict(
        test_common[features]
    )

    metrics = calculate_metrics(
        test_common[TARGET],
        predictions
    )

    results.append({
        "candidate": candidate,
        "n_features": len(features),
        "development_rows": len(
            development_common
        ),
        "test_rows": len(test_common),
        **metrics,
    })

    predictions_df[
        f"co2_predicted_{candidate}"
    ] = predictions

    _project_print(
        f"{candidate:25s} | "
        f"RMSE={metrics['rmse']:.4f} | "
        f"MAE={metrics['mae']:.4f} | "
        f"R2={metrics['r2']:.4f}"
    )

metrics_df = pd.DataFrame(results)

assert len(metrics_df) == len(candidate_names)

# ------------------------------------------------------------
# 8. GANHO SOBRE O BASELINE
# ------------------------------------------------------------

baseline_rmse = float(
    metrics_df.loc[
        metrics_df["candidate"] == "baseline",
        "rmse"
    ].iloc[0]
)

metrics_df["baseline_rmse"] = baseline_rmse

metrics_df["rmse_gain_vs_baseline_pct"] = (
    100
    * (
        baseline_rmse - metrics_df["rmse"]
    )
    / baseline_rmse
)

# ------------------------------------------------------------
# 9. COMPARAÇÃO COM O MODELO COMPLETO
# ------------------------------------------------------------

full_rmse = float(
    metrics_df.loc[
        metrics_df["candidate"] == "full_72",
        "rmse"
    ].iloc[0]
)

metrics_df["full_72_rmse"] = full_rmse

metrics_df["rmse_excess_vs_full_pct"] = (
    100
    * (
        metrics_df["rmse"] - full_rmse
    )
    / full_rmse
)

metrics_df["is_pretest_selected"] = (
    metrics_df["candidate"]
    == selected_candidate
)

# ------------------------------------------------------------
# 10. TABELA FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 72)
_project_print("RESULTADOS FINAIS — TESTE INDEPENDENTE")
_project_print("=" * 72)

_project_print(
    metrics_df[
        [
            "candidate",
            "n_features",
            "test_rows",
            "rmse",
            "mae",
            "r2",
            "bias",
            "rmse_gain_vs_baseline_pct",
            "rmse_excess_vs_full_pct",
            "is_pretest_selected",
        ]
    ].round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 11. GRÁFICO — OBSERVADO VS. ESTIMADO
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(
    predictions_df["timestamp"],
    predictions_df["co2_observed_metric_tons"],
    label="CO2 observado",
    linewidth=1.6
)

for candidate in [
    selected_candidate,
    "full_72",
]:

    ax.plot(
        predictions_df["timestamp"],
        predictions_df[
            f"co2_predicted_{candidate}"
        ],
        label=f"Estimado — {candidate}",
        linewidth=1.1,
        alpha=0.85
    )

ax.set_xlabel("Data")
ax.set_ylabel("CO2 (toneladas métricas/h)")

ax.set_title(
    "Teste retrospectivo — emissões observadas e estimadas"
)

ax.grid(alpha=0.25)
ax.legend()

plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE_SERIES,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 12. GRÁFICO — COMPARAÇÃO DE RMSE
# ------------------------------------------------------------

plot_df = metrics_df.sort_values(
    "rmse"
).copy()

fig, ax = plt.subplots(figsize=(11, 5))

ax.barh(
    plot_df["candidate"],
    plot_df["rmse"]
)

ax.set_xlabel("RMSE (toneladas métricas de CO2/h)")
ax.set_ylabel("Modelo")

ax.set_title(
    "Comparação final — conjunto de teste retrospectivo"
)

ax.invert_yaxis()
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()

plt.savefig(
    OUTPUT_FIGURE_COMPARISON,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 13. EXPORTAÇÃO
# ------------------------------------------------------------

_project_csv(metrics_df, 
    OUTPUT_METRICS,
    index=False
)

predictions_df.to_parquet(
    OUTPUT_PREDICTIONS,
    index=False
)

elapsed = time.time() - start_time

selected_metrics = metrics_df.loc[
    metrics_df["candidate"] == selected_candidate
].iloc[0]

audit = {
    "cell": "13/14",
    "status": "final_retrospective_test_completed",
    "unit": "Iatan Unit 2",
    "orispl_code": 6065,
    "year": 2025,
    "selected_candidate_pretest": selected_candidate,
    "candidate_names": candidate_names,
    "feature_sets": feature_sets,
    "model_parameters": rf_params,
    "development_rows_common": int(
        len(development_common)
    ),
    "test_rows_common": int(
        len(test_common)
    ),
    "test_start": str(
        test_common["timestamp"].min()
    ),
    "test_end": str(
        test_common["timestamp"].max()
    ),
    "selected_model_test_metrics": {
        "rmse": float(
            selected_metrics["rmse"]
        ),
        "mae": float(
            selected_metrics["mae"]
        ),
        "r2": float(
            selected_metrics["r2"]
        ),
        "bias": float(
            selected_metrics["bias"]
        ),
        "rmse_gain_vs_baseline_pct": float(
            selected_metrics[
                "rmse_gain_vs_baseline_pct"
            ]
        ),
    },
    "test_used": True,
    "posttest_reselection_performed": False,
    "execution_seconds": round(elapsed, 2),
}

with open(
    OUTPUT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        audit,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 14. INTEGRIDADE DOS ARQUIVOS
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


for path in [
    OUTPUT_METRICS,
    OUTPUT_PREDICTIONS,
    OUTPUT_AUDIT,
    OUTPUT_FIGURE_SERIES,
    OUTPUT_FIGURE_COMPARISON,
]:

    assert path.exists()

    _project_print(
        f"\nArquivo: {path.name}"
        f"\nSHA-256: {sha256_file(path)}"
    )

# ------------------------------------------------------------
# 15. RESUMO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 72)

_project_print(
    "CÉLULA 13/14 CONCLUÍDA — "
    "TESTE FINAL INDEPENDENTE"
)

_project_print("=" * 72)

_project_print(
    "Modelo selecionado antes do teste:",
    selected_candidate
)

_project_print(
    "RMSE no teste:",
    round(selected_metrics["rmse"], 4)
)

_project_print(
    "Ganho sobre baseline:",
    round(
        selected_metrics[
            "rmse_gain_vs_baseline_pct"
        ],
        2
    ),
    "%"
)

_project_print(
    "Observações de teste:",
    len(test_common)
)

_project_print(
    "Tempo de execução:",
    round(elapsed, 1),
    "s"
)

_project_print("Teste utilizado uma única vez.")
_project_print("Seleção pré-teste preservada.")
_project_print("Nenhuma nova seleção realizada.")

_project_print(
    "PRÓXIMA CÉLULA: 14 — "
    "exportação e documentação final."
)


In [ ]:
# Referência metodológica: Aula 08, pp. 31–32 e 53–56. Registro de resultados, limitações, hashes e pacote de evidências.

# ============================================================
# CÉLULA 14/14 — DOCUMENTAÇÃO E EXPORTAÇÃO FINAL
# ============================================================
#
# PROJETO:
# Seleção preditiva de lags para estimação horária de CO2
#
# UNIDADE:
# Iatan — Unit 2 — ORISPL 6065 — 2025
#
# OBJETIVOS:
# 1. Consolidar os resultados das Células 06–13.
# 2. Documentar a seleção pré-teste e o teste final.
# 3. Auditar consistência dos resultados.
# 4. Produzir relatório técnico em Markdown.
# 5. Exportar tabelas, manifesto e arquivos de integridade.
#
# RESTRIÇÕES:
# - Nenhum treinamento adicional.
# - Nenhuma alteração dos hiperparâmetros.
# - Nenhuma nova seleção de variáveis.
# - Nenhuma reavaliação do conjunto de teste.
# - Preservação dos arquivos anteriores.
#
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------


REPORT_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"
DATA_DIR = RUN_DIR / "data/processed"

FINAL_DIR = RUN_DIR / "final"
FINAL_TABLES = FINAL_DIR / "tables"
FINAL_FIGURES = FINAL_DIR / "figures"
FINAL_DOCS = FINAL_DIR / "documentation"

for directory in [
    FINAL_DIR,
    FINAL_TABLES,
    FINAL_FIGURES,
    FINAL_DOCS,
]:
    directory.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "baseline_metrics":
        REPORT_DIR / "cell06_baseline_metrics.csv",

    "horizon_comparison":
        REPORT_DIR / "cell07_horizon_comparison.csv",

    "lag_importance":
        REPORT_DIR / "cell08_lag_importance.csv",

    "lag_redundancy":
        REPORT_DIR / "cell08_lag_redundancy.csv",

    "lag_selection":
        REPORT_DIR / "cell09_parsimonious_lag_comparison.csv",

    "rolling_comparison":
        REPORT_DIR / "cell10_rolling_window_comparison.csv",

    "consolidated_comparison":
        REPORT_DIR / "cell11_consolidated_comparison.csv",

    "robustness_folds":
        REPORT_DIR / "cell12_temporal_robustness_folds.csv",

    "robustness_summary":
        REPORT_DIR / "cell12_temporal_robustness_summary.csv",

    "pretest_decision":
        REPORT_DIR / "cell12_pretest_model_decision.json",

    "final_test_metrics":
        REPORT_DIR / "cell13_final_test_metrics.csv",

    "final_test_audit":
        REPORT_DIR / "cell13_final_test_audit.json",

    "final_test_predictions":
        DATA_DIR / "cell13_final_test_predictions.parquet",
}

for name, path in INPUTS.items():
    assert path.exists(), (
        f"Arquivo obrigatório ausente: {name} -> {path}"
    )

_project_print("Arquivos obrigatórios encontrados:", len(INPUTS))

# ------------------------------------------------------------
# 2. FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        _project_dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def fmt(value, decimals=4):
    return f"{float(value):.{decimals}f}"


def markdown_table(df):
    columns = list(df.columns)
    lines = [
        "| " + " | ".join(map(str, columns)) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]

    for row in df.itertuples(index=False, name=None):
        lines.append(
            "| " + " | ".join(
                str(value).replace("|", "\\|")
                for value in row
            ) + " |"
        )

    return "\n".join(lines)


# ------------------------------------------------------------
# 3. CARREGAMENTO DOS RESULTADOS
# ------------------------------------------------------------

baseline = pd.read_csv(INPUTS["baseline_metrics"])
horizons = pd.read_csv(INPUTS["horizon_comparison"])
importance = pd.read_csv(INPUTS["lag_importance"])
redundancy = pd.read_csv(INPUTS["lag_redundancy"])
lag_selection = pd.read_csv(INPUTS["lag_selection"])
rolling = pd.read_csv(INPUTS["rolling_comparison"])
consolidated = pd.read_csv(
    INPUTS["consolidated_comparison"]
)
robustness_folds = pd.read_csv(
    INPUTS["robustness_folds"]
)
robustness = pd.read_csv(
    INPUTS["robustness_summary"]
)
decision = load_json(INPUTS["pretest_decision"])
test_metrics = pd.read_csv(
    INPUTS["final_test_metrics"]
)
test_audit = load_json(INPUTS["final_test_audit"])

# Não é necessário carregar as previsões de teste:
# o arquivo será preservado e incluído no pacote final.

# ------------------------------------------------------------
# 4. AUDITORIA DA DECISÃO PRÉ-TESTE
# ------------------------------------------------------------

selected_candidate = decision["selected_candidate"]

assert decision["selection_status"] == "pretest_frozen"
assert decision["test_used"] is False

assert selected_candidate == "compact_8"

assert (
    test_audit["selected_candidate_pretest"]
    == selected_candidate
)

assert test_audit["test_used"] is True

assert (
    test_audit["posttest_reselection_performed"]
    is False
)

assert (
    test_metrics["is_pretest_selected"].sum() == 1
)

assert (
    test_metrics.loc[
        test_metrics["is_pretest_selected"],
        "candidate"
    ].iloc[0] == selected_candidate
)

assert test_metrics["test_rows"].nunique() == 1

assert int(test_metrics["test_rows"].iloc[0]) == 710

assert (
    test_metrics["development_rows"].nunique() == 1
)

assert (
    int(test_metrics["development_rows"].iloc[0])
    == 3409
)

_project_print("Auditoria da decisão pré-teste: OK")

# ------------------------------------------------------------
# 5. CONSISTÊNCIA MATEMÁTICA DAS MÉTRICAS
# ------------------------------------------------------------

baseline_test_rmse = float(
    test_metrics.loc[
        test_metrics["candidate"] == "baseline",
        "rmse"
    ].iloc[0]
)

calculated_gain = (
    100
    * (
        baseline_test_rmse - test_metrics["rmse"]
    )
    / baseline_test_rmse
)

assert np.allclose(
    calculated_gain,
    test_metrics["rmse_gain_vs_baseline_pct"],
    atol=1e-6,
    rtol=1e-6,
)

assert (
    test_metrics["candidate"].nunique() == 4
)

assert set(test_metrics["candidate"]) == {
    "baseline",
    "compact_8",
    "compact_8_rolling_3_6",
    "full_72",
}

_project_print("Auditoria matemática das métricas: OK")

# ------------------------------------------------------------
# 6. TABELA FINAL DO TESTE
# ------------------------------------------------------------

test_table = test_metrics[
    [
        "candidate",
        "n_features",
        "test_rows",
        "rmse",
        "mae",
        "r2",
        "bias",
        "rmse_gain_vs_baseline_pct",
        "is_pretest_selected",
    ]
].copy()

test_table = test_table.round(4)

_project_csv(test_table, 
    FINAL_TABLES / "final_test_results.csv",
    index=False,
)

# ------------------------------------------------------------
# 7. TABELA DE ROBUSTEZ
# ------------------------------------------------------------

robustness_table = robustness[
    [
        "candidate",
        "n_features",
        "folds",
        "rmse_mean",
        "rmse_std",
        "mae_mean",
        "gain_mean_pct",
        "rmse_cv_pct",
    ]
].copy()

robustness_table = robustness_table.round(4)

_project_csv(robustness_table, 
    FINAL_TABLES / "temporal_robustness_results.csv",
    index=False,
)

# ------------------------------------------------------------
# 8. COMPARAÇÃO DESENVOLVIMENTO × TESTE
# ------------------------------------------------------------
#
# Observação:
# Os RMSEs médios dos folds e os RMSEs do teste
# pertencem a períodos distintos.
#
# A comparação é descritiva; não se deve interpretar
# a diferença absoluta como melhoria temporal
# de um mesmo modelo em uma mesma distribuição.
# ------------------------------------------------------------

comparison = robustness[
    ["candidate", "n_features", "rmse_mean", "gain_mean_pct"]
].merge(
    test_metrics[
        [
            "candidate",
            "rmse",
            "rmse_gain_vs_baseline_pct",
        ]
    ],
    on="candidate",
    how="inner",
    validate="one_to_one",
)

comparison = comparison.rename(
    columns={
        "rmse_mean": "robustness_rmse_mean",
        "gain_mean_pct": "robustness_gain_mean_pct",
        "rmse": "test_rmse",
        "rmse_gain_vs_baseline_pct":
            "test_gain_vs_baseline_pct",
    }
)

comparison["gain_change_percentage_points"] = (
    comparison["test_gain_vs_baseline_pct"]
    - comparison["robustness_gain_mean_pct"]
)

comparison = comparison.round(4)

_project_csv(comparison, 
    FINAL_TABLES / "development_vs_test.csv",
    index=False,
)

# ------------------------------------------------------------
# 9. AUDITORIA DA COBERTURA DO TESTE
# ------------------------------------------------------------

eligible_test_rows = 1063
common_test_rows = int(
    test_metrics["test_rows"].iloc[0]
)

coverage_pct = (
    100 * common_test_rows / eligible_test_rows
)

excluded_due_to_common_history = (
    eligible_test_rows - common_test_rows
)

coverage = {
    "eligible_test_rows": eligible_test_rows,
    "common_test_rows": common_test_rows,
    "common_test_coverage_pct": coverage_pct,
    "excluded_from_common_comparison":
        excluded_due_to_common_history,
    "reason": (
        "Common comparison requires observed "
        "G(t), CO2(t) and all 72 generation lags."
    ),
}

write_json(
    FINAL_DOCS / "test_coverage.json",
    coverage,
)

# ------------------------------------------------------------
# 10. RESULTADOS CENTRAIS
# ------------------------------------------------------------

selected_row = test_metrics.loc[
    test_metrics["candidate"] == selected_candidate
].iloc[0]

full_row = test_metrics.loc[
    test_metrics["candidate"] == "full_72"
].iloc[0]

baseline_row = test_metrics.loc[
    test_metrics["candidate"] == "baseline"
].iloc[0]

selected_robustness = robustness.loc[
    robustness["candidate"] == selected_candidate
].iloc[0]

selected_rmse = float(selected_row["rmse"])
selected_gain = float(
    selected_row["rmse_gain_vs_baseline_pct"]
)

full_rmse = float(full_row["rmse"])

baseline_is_best = bool(
    baseline_test_rmse
    == test_metrics["rmse"].min()
)

assert baseline_is_best

assert selected_gain < 0

# ------------------------------------------------------------
# 11. FIGURA DESCRITIVA DE GANHOS
# ------------------------------------------------------------

plot_df = comparison.loc[
    comparison["candidate"] != "baseline"
].copy()

x = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots(figsize=(11, 6))

ax.bar(
    x - width / 2,
    plot_df["robustness_gain_mean_pct"],
    width,
    label="Robustez temporal",
)

ax.bar(
    x + width / 2,
    plot_df["test_gain_vs_baseline_pct"],
    width,
    label="Teste retrospectivo",
)

ax.axhline(0, linewidth=1)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["candidate"],
    rotation=15,
    ha="right",
)

ax.set_ylabel(
    "Redução relativa do RMSE vs. baseline (%)"
)

ax.set_title(
    "Ganhos relativos — desenvolvimento e teste"
)

ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

FIG_GAIN = (
    FINAL_FIGURES
    / "development_vs_test_relative_gains.png"
)

plt.savefig(
    FIG_GAIN,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

# ------------------------------------------------------------
# 12. REGISTRO DAS LIMITAÇÕES
# ------------------------------------------------------------

limitations = [
    (
        "A análise utiliza uma única unidade termelétrica "
        "e um único ano de observações."
    ),
    (
        "Os lags foram escolhidos durante o desenvolvimento. "
        "A validação temporal posterior não é independente "
        "desse processo de seleção."
    ),
    (
        "O protocolo congelou a seleção antes da avaliação. "
        "Como o período de teste já foi consultado no projeto, esta avaliação é retrospectiva."
    ),
    (
        "A comparação principal utiliza 710 das 1063 "
        "observações elegíveis de teste, exigindo "
        "histórico completo de 72 horas."
    ),
    (
        "A avaliação não quantifica a cobertura adicional "
        "que representações compactas poderiam alcançar "
        "em horas com histórico incompleto."
    ),
    (
        "As emissões são estimadas a partir da geração "
        "elétrica, sem consumo de combustível, composição "
        "do combustível ou eficiência operacional medida."
    ),
    (
        "O benefício preditivo dos lags observado durante "
        "o desenvolvimento não se confirmou no teste final."
    ),
    (
        "Os resultados não podem ser transferidos "
        "automaticamente para outras usinas ou para o "
        "Sistema Interligado Nacional."
    ),
    (
        "A figura temporal original do teste conecta "
        "observações separadas por lacunas no calendário; "
        "segmentos diagonais nessas regiões não representam "
        "necessariamente trajetórias operacionais reais."
    ),
]

# ------------------------------------------------------------
# 13. RELATÓRIO TÉCNICO EM MARKDOWN
# ------------------------------------------------------------

test_md = markdown_table(test_table)
robustness_md = markdown_table(robustness_table)
comparison_md = markdown_table(comparison)

limitations_md = "\n".join(
    f"{i}. {item}"
    for i, item in enumerate(limitations, start=1)
)

selected_features_md = "\n".join(
    f"- `{feature}`"
    for feature in decision["selected_features"]
)

report = f"""# Relatório técnico — Seleção preditiva de lags

## 1. Identificação

- Unidade: Iatan — Unit 2
- ORISPL: 6065
- Ano: 2025
- Fonte: EPA CEMS
- Resolução: horária
- Experimento: seleção de lags da geração elétrica
- Variável-alvo: emissões horárias de CO2 em toneladas métricas

## 2. Objetivo

Avaliar se a inclusão de defasagens temporais da geração
elétrica melhora a estimação contemporânea das emissões
horárias de CO2 em relação ao uso exclusivo de G(t).

## 3. Delineamento experimental

O experimento utilizou:

- Geração contemporânea G(t).
- Lags horários de geração, até 168 horas na exploração.
- Horizontes cumulativos de histórico.
- Importância por permutação.
- Avaliação de redundância entre preditores.
- Subconjuntos compactos de lags.
- Médias móveis de 3, 6, 12 e 24 horas.
- Modelos Ridge e Random Forest.
- Validação cronológica e robustez temporal.
- Conjunto de teste retrospectivo.

O modelo Random Forest foi utilizado como família principal
na etapa final, mantendo os hiperparâmetros previamente
fixados.

Dados ausentes não foram automaticamente convertidos
em geração ou emissão igual a zero.

## 4. Seleção antes do teste

Candidato prioritário: `{selected_candidate}`.

Número de preditores: {len(decision["selected_features"])}.

Variáveis:

{selected_features_md}

A seleção foi realizada antes da abertura do conjunto
de teste, utilizando uma tolerância de
{fmt(decision["tolerance_pct"], 1)}% em relação ao
menor RMSE médio da robustez temporal.

## 5. Robustez temporal

Foram utilizados quatro folds cronológicos progressivos.

{robustness_md}

O modelo compacto selecionado apresentou RMSE médio
de {fmt(selected_robustness["rmse_mean"])} e ganho
médio de {fmt(selected_robustness["gain_mean_pct"], 2)}%
em relação ao baseline.

## 6. Teste retrospectivo

Período reservado: 06/11/2025 a 31/12/2025.

Observações elegíveis: {eligible_test_rows}.

Observações na comparação comum: {common_test_rows}.

Cobertura da amostra comum:
{fmt(coverage_pct, 2)}%.

{test_md}

O baseline G(t) apresentou o menor RMSE no teste:
{fmt(baseline_test_rmse)} toneladas métricas de CO2/h.

O modelo compacto, selecionado antes do teste, apresentou
RMSE de {fmt(selected_rmse)} e variação relativa de
{fmt(selected_gain, 2)}% no indicador de redução do RMSE
em comparação ao baseline.

Como o ganho é negativo, o erro do modelo compacto foi
maior do que o erro do baseline.

O modelo completo com 72 lags apresentou RMSE de
{fmt(full_rmse)}, também sem superar o baseline.

## 7. Comparação descritiva

{comparison_md}

Os resultados mostram que os ganhos observados durante
o desenvolvimento não se mantiveram no teste retrospectivo.

As diferenças absolutas entre RMSEs de desenvolvimento
e teste não devem ser interpretadas isoladamente, pois
os períodos possuem condições e distribuições
potencialmente distintas.

## 8. Discussão

A geração elétrica contemporânea apresentou forte
capacidade preditiva para as emissões horárias desta
unidade no período de teste.

A inclusão de histórico de geração produziu ganhos
durante a validação, mas não melhorou o RMSE no teste
independente.

Esse resultado indica que o benefício de lags pode
depender das condições operacionais e da estabilidade
temporal das relações entre geração e emissões.

A presente análise não identifica causalmente as
razões dessa mudança de desempenho.

## 9. Limitações

{limitations_md}

## 10. Conclusão

A hipótese de que o histórico de geração melhora
consistentemente a estimação contemporânea de CO2
não foi confirmada no teste retrospectivo desta unidade.

Embora representações temporais tenham melhorado
o desempenho durante o desenvolvimento, o baseline
baseado exclusivamente em G(t) apresentou o menor
RMSE no período de teste.

O resultado não demonstra que lags sejam inúteis em
outras unidades ou períodos. Demonstra que sua
vantagem preditiva não foi robusta neste teste
independente específico.

## 11. Reprodutibilidade

Os resultados e decisões foram preservados nos
arquivos das Células 06–13.

A Célula 14 não realiza novos treinamentos, não
altera hiperparâmetros e não reutiliza o teste
para selecionar modelos.

O pacote final inclui tabelas, figuras,
configurações, auditoria e manifesto de integridade.
"""

REPORT_PATH = (
    FINAL_DOCS / "technical_report.md"
)

_project_write_text(REPORT_PATH, 
    report,
    encoding="utf-8",
)

_project_print("\nRELATÓRIO TÉCNICO")
_project_print("Arquivo:", REPORT_PATH)
_project_print("Caracteres:", len(report))

# ------------------------------------------------------------
# 14. CÓPIA DOS RESULTADOS ORIGINAIS
# ------------------------------------------------------------
#
# Os arquivos originais permanecem intactos.
#
# Copiamos somente resultados necessários
# para reconstituir a análise final.
# ------------------------------------------------------------

SOURCE_TABLES = [
    INPUTS["baseline_metrics"],
    INPUTS["horizon_comparison"],
    INPUTS["lag_importance"],
    INPUTS["lag_redundancy"],
    INPUTS["lag_selection"],
    INPUTS["rolling_comparison"],
    INPUTS["consolidated_comparison"],
    INPUTS["robustness_folds"],
    INPUTS["robustness_summary"],
    INPUTS["final_test_metrics"],
]

SOURCE_DOCS = [
    INPUTS["pretest_decision"],
    INPUTS["final_test_audit"],
]

for source in SOURCE_TABLES:
    shutil.copy2(
        source,
        FINAL_TABLES / source.name,
    )

for source in SOURCE_DOCS:
    shutil.copy2(
        source,
        FINAL_DOCS / source.name,
    )

# Preservar previsões finais no pacote.
shutil.copy2(
    INPUTS["final_test_predictions"],
    FINAL_TABLES
    / INPUTS["final_test_predictions"].name,
)

# Figuras disponíveis das etapas anteriores.
figure_names = [
    "cell08_lag_importance.png",
    "cell08_lag_redundancy_heatmap.png",
    "cell09_parsimonious_lag_comparison.png",
    "cell10_rolling_window_comparison.png",
    "cell11_performance_complexity.png",
    "cell12_temporal_robustness.png",
    "cell13_observed_vs_predicted.png",
    "cell13_final_test_comparison.png",
]

missing_optional_figures = []

for name in figure_names:

    source = FIG_DIR / name

    if source.exists():
        shutil.copy2(
            source,
            FINAL_FIGURES / name,
        )
    else:
        missing_optional_figures.append(name)

# ------------------------------------------------------------
# 15. CORREÇÃO EXCLUSIVAMENTE VISUAL DAS LACUNAS
# ------------------------------------------------------------
#
# A figura da Célula 13 ligava observações
# separadas por lacunas no calendário.
#
# Aqui, reconstruímos apenas a visualização,
# sem recalcular previsões ou métricas.
#
# Segmentos são separados quando o intervalo
# entre timestamps consecutivos excede 1 hora.
# ------------------------------------------------------------

pred = pd.read_parquet(
    INPUTS["final_test_predictions"]
)

pred["timestamp"] = pd.to_datetime(
    pred["timestamp"],
    errors="raise",
)

pred = pred.sort_values(
    "timestamp"
).reset_index(drop=True)

assert len(pred) == common_test_rows

gap = (
    pred["timestamp"].diff()
    > pd.Timedelta(hours=1)
)

segment_id = gap.cumsum()

fig, ax = plt.subplots(figsize=(15, 6))

series_columns = {
    "co2_observed_metric_tons": "CO2 observado",
    f"co2_predicted_{selected_candidate}":
        f"Estimado — {selected_candidate}",
    "co2_predicted_full_72":
        "Estimado — full_72",
}

for column, label in series_columns.items():

    first_segment = True

    for _, segment in pred.groupby(segment_id):

        ax.plot(
            segment["timestamp"],
            segment[column],
            linewidth=1.2,
            label=label if first_segment else None,
        )

        first_segment = False

ax.set_title(
    "Teste retrospectivo — séries com lacunas preservadas"
)

ax.set_xlabel("Data")
ax.set_ylabel("CO2 (toneladas métricas/h)")
ax.grid(alpha=0.25)
ax.legend()

plt.xticks(rotation=30)
plt.tight_layout()

FIG_CORRECTED = (
    FINAL_FIGURES
    / "final_test_series_without_gap_connections.png"
)

plt.savefig(
    FIG_CORRECTED,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

# ------------------------------------------------------------
# 16. MANIFESTO DE INTEGRIDADE
# ------------------------------------------------------------
#
# O manifesto lista os arquivos finais,
# seus tamanhos e hashes SHA-256.
#
# O manifesto e o ZIP são criados depois,
# para evitar referências circulares.
# ------------------------------------------------------------

MANIFEST_PATH = (
    FINAL_DIR / "manifest_sha256.csv"
)

ZIP_PATH = (
    RUN_DIR / "lag_selection_predictive_final.zip"
)

# Evitar resíduos de execução anterior no manifesto.
if MANIFEST_PATH.exists():
    MANIFEST_PATH.unlink()

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

manifest_rows = []

for path in sorted(FINAL_DIR.rglob("*")):

    if not path.is_file():
        continue

    manifest_rows.append({
        "relative_path": str(
            path.relative_to(FINAL_DIR)
        ),
        "size_bytes": int(
            path.stat().st_size
        ),
        "sha256": sha256_file(path),
    })

manifest = pd.DataFrame(manifest_rows)

_project_csv(manifest, 
    MANIFEST_PATH,
    index=False,
)

# ------------------------------------------------------------
# 17. AUDITORIA FINAL
# ------------------------------------------------------------

final_audit = {
    "cell": "14/14",
    "status": "experiment_documented",
    "unit": "Iatan Unit 2",
    "orispl_code": 6065,
    "year": 2025,
    "selected_pretest_candidate": selected_candidate,
    "selected_pretest_n_features": int(
        selected_row["n_features"]
    ),
    "baseline_test_rmse": baseline_test_rmse,
    "selected_test_rmse": selected_rmse,
    "selected_test_gain_pct": selected_gain,
    "full_72_test_rmse": full_rmse,
    "eligible_test_rows": eligible_test_rows,
    "common_test_rows": common_test_rows,
    "common_test_coverage_pct": coverage_pct,
    "baseline_lowest_test_rmse": baseline_is_best,
    "posttest_reselection_performed": False,
    "new_training_performed": False,
    "test_recomputed": False,
    "report_path": str(REPORT_PATH),
    "manifest_path": str(MANIFEST_PATH),
    "missing_optional_figures":
        missing_optional_figures,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

FINAL_AUDIT_PATH = (
    RUN_DIR / "cell14_final_audit.json"
)

write_json(
    FINAL_AUDIT_PATH,
    final_audit,
)

# ------------------------------------------------------------
# 18. PACOTE ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:

    for path in sorted(FINAL_DIR.rglob("*")):

        if path.is_file():

            archive.write(
                path,
                arcname=str(
                    path.relative_to(FINAL_DIR)
                ),
            )

    archive.write(
        FINAL_AUDIT_PATH,
        arcname=FINAL_AUDIT_PATH.name,
    )

# ------------------------------------------------------------
# 19. VERIFICAÇÃO DO PACOTE
# ------------------------------------------------------------

assert ZIP_PATH.exists()
assert ZIP_PATH.stat().st_size > 0

with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    bad_member = archive.testzip()

    assert bad_member is None, (
        f"Arquivo ZIP corrompido: {bad_member}"
    )

    names = set(archive.namelist())

    assert (
        "documentation/technical_report.md"
        in names
    )

    assert (
        "tables/final_test_results.csv"
        in names
    )

    assert (
        "manifest_sha256.csv"
        in names
    )

_project_print("\nPACOTE FINAL")
_project_print("Diretório:", FINAL_DIR)
_project_print("ZIP:", ZIP_PATH)

_project_print(
    "Tamanho ZIP (MB):",
    round(
        ZIP_PATH.stat().st_size / 1024**2,
        2,
    )
)

_project_print(
    "SHA-256 ZIP:",
    sha256_file(ZIP_PATH)
)

_project_print(
    "Arquivos no manifesto:",
    len(manifest)
)

# ------------------------------------------------------------
# 20. RESUMO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 72)

_project_print(
    "CÉLULA 14/14 CONCLUÍDA — "
    "EXPERIMENTO DOCUMENTADO"
)

_project_print("=" * 72)

_project_print("Unidade: Iatan — Unit 2")
_project_print("Ano: 2025")

_project_print(
    "Modelo selecionado antes do teste:",
    selected_candidate
)

_project_print(
    "RMSE do baseline no teste:",
    round(baseline_test_rmse, 4)
)

_project_print(
    "RMSE do modelo selecionado:",
    round(selected_rmse, 4)
)

_project_print(
    "Ganho do modelo selecionado:",
    round(selected_gain, 2),
    "%"
)

_project_print(
    "RMSE do modelo completo:",
    round(full_rmse, 4)
)

_project_print(
    "Cobertura da comparação comum:",
    round(coverage_pct, 2),
    "%"
)

_project_print("\nNenhum novo treinamento realizado.")
_project_print("Nenhuma seleção pós-teste realizada.")
_project_print("Resultados originais preservados.")
_project_print("Relatório técnico gerado.")
_project_print("Manifesto SHA-256 gerado.")
_project_print("Pacote ZIP verificado.")

_project_print("\nEXPERIMENTO CONCLUÍDO: 14/14 CÉLULAS.")


### Resultado histórico para leitura

Na comparação original sobre as 710 horas de teste em comum, os RMSEs registrados foram 31,0960 para a geração contemporânea, 33,9762 para a representação compacta escolhida durante o desenvolvimento e 31,2880 t CO₂/h para 72 lags. Os ganhos no desenvolvimento não se mantiveram nesse recorte de avaliação. A cobertura por idade dos ciclos, calculada na auditoria abaixo, ajuda a delimitar a interpretação. O teste já foi consultado pelo grupo. A validação temporal e a separação entre seleção e avaliação seguem a motivação de [Cerqueira et al.](https://arxiv.org/abs/1905.11744) e [Cawley e Talbot](https://jmlr.org/papers/v11/cawley10a.html).


## Auditorias complementares

V2-A refaz o ranking de lags apenas dentro do treino de cada janela expansiva. Usa importância por impureza e outra configuração de Random Forest, enquanto a seleção principal usa permutação na validação. A comparação é diagnóstica. V2-B descreve a disponibilidade do histórico por partição e idade de ciclo, sem selecionar representações. Os nomes dos CSVs antigos foram preservados para compatibilidade.


In [ ]:
# Referência metodológica: Breiman (2001) e Cerqueira et al. (2020). Reordenação dos lags no treino de cada janela expansiva.
# V2-A — Janelas expansivas e ranking por treino (somente treino + validação)
# Seleção dos lags é refeita DENTRO de cada fold, sem olhar o bloco validado.
# Avaliação de estimativa contemporânea, não previsão de CO2 futuro.
import numpy as np, pandas as pd, json
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from IPython.display import display

v2_root = Path(RUN_DIR) / "reports" / "phase3_v2_audit"
v2_root.mkdir(parents=True, exist_ok=True)
lag_source = RUN_DIR / "data/processed/cell04_generation_lag_bank_168h.parquet"
if not lag_source.is_file():
    raise FileNotFoundError(f"Execute a Célula 04 da mesma execução: {lag_source}")
d = pd.read_parquet(lag_source).copy()
d["timestamp"] = pd.to_datetime(d["timestamp"])
d = d.loc[d["partition"].isin(["train", "validation"]) & d["eligible_for_modeling"].eq(True)].sort_values("timestamp").reset_index(drop=True)
lagcols = [f"generation_lag_{k:03d}h" for k in range(1,73)]
basecol = "generation_mw"; target = "co2_metric_tons"
need = [basecol,target]+lagcols
missing = sorted(set(need)-set(d.columns))
if missing: raise ValueError(f"Colunas ausentes: {missing[:8]}")
# Amostra comum permite comparação honesta de representações de até 72 h.
d = d.dropna(subset=need).reset_index(drop=True)
if len(d) < 400: raise ValueError("Amostra insuficiente para auditoria em 4 folds")
initial = int(len(d)*0.50)
blocks = [x for x in np.array_split(np.arange(initial,len(d)),4) if len(x)]
rows=[]; selected=[]
for f,idx in enumerate(blocks,1):
    tr=d.iloc[:int(idx[0])].copy(); va=d.iloc[idx].copy()
    if len(tr)<100 or len(va)<20: continue
    # Etapa interna: ranking de lags apenas no treino do fold.
    selector=RandomForestRegressor(n_estimators=120,min_samples_leaf=5,max_features=0.7,n_jobs=-1,random_state=420+f)
    selector.fit(tr[[basecol]+lagcols],tr[target])
    importance=pd.Series(selector.feature_importances_[1:],index=lagcols).sort_values(ascending=False)
    top8=importance.head(8).index.tolist()
    selected.append({"fold":f,"selected_lags":_project_dumps(top8),"validation_start":str(va.timestamp.min()),"validation_end":str(va.timestamp.max())})
    for label,cols in [("baseline",[basecol]),("nested_top8",[basecol]+top8),("full_72",[basecol]+lagcols)]:
        model=RandomForestRegressor(n_estimators=180,min_samples_leaf=5,max_features=0.7,n_jobs=-1,random_state=2026)
        model.fit(tr[cols],tr[target]); pred=model.predict(va[cols])
        rows.append({"fold":f,"candidate":label,"train_n":len(tr),"validation_n":len(va),"validation_start":str(va.timestamp.min()),"validation_end":str(va.timestamp.max()),"rmse":float(np.sqrt(mean_squared_error(va[target],pred))),"mae":float(mean_absolute_error(va[target],pred))})
fold_v2=pd.DataFrame(rows); lag_selection_v2=pd.DataFrame(selected)
if fold_v2.empty: raise RuntimeError("Nenhum fold concluído")
_project_csv(fold_v2, v2_root/"nested_temporal_folds.csv",index=False)
_project_csv(lag_selection_v2, v2_root/"nested_lag_selection.csv",index=False)
_project_print("Auditoria exploratória: ranking de lags dentro do treino de cada janela.")
_project_display(fold_v2)
_project_display(lag_selection_v2)


In [ ]:
# Referência metodológica: EPA CAMD Data Guide e Aula 08. Operating Time descreve operação. Idade e cobertura são diagnósticos do projeto.
# V2-B — Cobertura do histórico e idade dos ciclos observados
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

v2_root = RUN_DIR / "reports/phase3_v2_audit"
v2_root.mkdir(parents=True, exist_ok=True)
src = pd.read_parquet(RUN_DIR / "data/processed/cell04_generation_lag_bank_168h.parquet")
src = src.sort_values("timestamp").reset_index(drop=True)
cols = [f"generation_lag_{k:03d}h" for k in range(1, 73)]
required = ["timestamp", "partition", "eligible_for_modeling", "generation_mw", "co2_metric_tons"] + cols
missing = set(required) - set(src.columns)
if missing:
    raise ValueError(f"Colunas ausentes: {sorted(missing)}")

# Idade operacional usada somente no diagnóstico, sem entrar nos modelos.
inventory = json.loads((RUN_DIR / "reports/cell01_configuration_inventory.json").read_text(encoding="utf-8"))
source_record = inventory["inputs"]["phase1_hourly_generation_co2"]
op_path = Path(source_record["path"])
if sha256_file(op_path) != source_record["sha256"]:
    raise RuntimeError("A série horária mudou desde o inventário desta execução.")
operation = pd.read_parquet(op_path, columns=["timestamp", "Operating Time"])
operation["timestamp"] = pd.to_datetime(operation["timestamp"], errors="raise")
src = src.merge(operation, on="timestamp", how="left", validate="one_to_one")
if not src["Operating Time"].between(0, 1).all():
    raise ValueError("Operating Time ausente ou fora do intervalo [0, 1].")
on = src["Operating Time"].gt(0)
cycle = (on & ~on.shift(fill_value=False)).cumsum()
src["cycle_age_h"] = src.loc[on].groupby(cycle[on]).cumcount().add(1)

src = src.loc[src["eligible_for_modeling"].eq(True)].copy()
if src["cycle_age_h"].isna().any():
    raise ValueError("Há horas elegíveis fora dos ciclos ON. Revise o contrato do insumo.")
src["complete_72h"] = src[["generation_mw", "co2_metric_tons"] + cols].notna().all(axis=1)
src["cycle_age_band"] = pd.cut(
    src["cycle_age_h"], bins=[0, 1, 6, 24, 72, float("inf")],
    labels=["1", "2–6", "7–24", "25–72", "73+"],
)

def coverage_table(by):
    grouped = src.groupby(by, dropna=False, observed=True).agg(
        eligible_hours=("complete_72h", "size"),
        complete_72h_hours=("complete_72h", "sum"),
        zero_generation_hours=("generation_mw", lambda x: int(x.eq(0).sum())),
    ).reset_index()
    grouped["excluded_hours"] = grouped["eligible_hours"] - grouped["complete_72h_hours"]
    grouped["coverage_pct"] = 100 * grouped["complete_72h_hours"] / grouped["eligible_hours"]
    return grouped

coverage = coverage_table("partition")
coverage_by_age = coverage_table(["partition", "cycle_age_band"])
_project_csv(coverage, v2_root / "coverage_72h_by_partition.csv", index=False)
_project_csv(coverage_by_age, v2_root / "coverage_72h_by_cycle_age.csv", index=False)
_project_display(coverage)
_project_display(coverage_by_age)

manifest = {
    "project": "Iatan unit 2", "year": 2025, "run_id": RUN_ID,
    "version": "phase3_scientific_review",
    "test_status": "retrospective_previously_examined",
    "cycle_age_definition": "calendar hours since Operating Time > 0, counted before eligibility filtering",
    "selection_protocol": "V2-A ranks lags on the training segment of each expanding window",
    "caveat": "Coverage is descriptive and does not retune models. The 72 h complete-case sample can omit early cycle hours.",
    "outputs": ["coverage_72h_by_partition.csv", "coverage_72h_by_cycle_age.csv"],
    "ranking_artifacts": [name for name in ("nested_temporal_folds.csv", "nested_lag_selection.csv")
                          if (v2_root / name).is_file()],
}
_project_write_text(v2_root / "audit_manifest.json", _project_dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
_project_print("Relatórios salvos em:", v2_root)


## Fontes vinculadas às células

| Células do fluxo | Fonte e uso |
|---|---|
| 01, 11 e 14 | **IND2622, Aula 08 — e-Science, método científico e DoE**, pp. 8–9, 31–32 e 53–56. Plano, log de resultados e proveniência. PDF no acervo do projeto. |
| 02–03 e V2-B | **US EPA, CAMD’s Power Sector Emissions Data Guide**, pp. 1–2. Origem e unidade da massa e geração. [Portal oficial](https://www.epa.gov/power-sector/cam-api-portal). |
| 04, 07 e 10 | **Gu, Sward e Zhang (2023), Predicting power plant emissions using public data and machine learning**, § 2.4. Contexto da validação temporal de modelos CEMS, em outro desenho de estudo. A grade, as defasagens e a cobertura deste trabalho são decisões próprias. [Artigo](https://doi.org/10.1039/D3VA00191A). [pandas shift](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shift.html) e [rolling](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html). |
| 05 | **statsmodels**, [documentação da PACF](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.pacf.html). A ACF e a CCF descritivas usam pares de horas disponíveis no calendário. |
| 06–10, 12–13 e V2-A | **Breiman (2001), Random Forests** ([PDF do autor](https://www.stat.berkeley.edu/~breiman/randomforest2001.pdf)). [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) e [importância por permutação](https://scikit-learn.org/stable/modules/permutation_importance.html), documentação do scikit-learn. |
| 06–07, 12–14 | **Hyndman e Koehler (2006), Another look at measures of forecast accuracy**. Comparação de métricas de erro, em unidades de t CO₂/h neste estudo. [Artigo](https://doi.org/10.1016/j.ijforecast.2006.03.001). |
| 03, 07–09, 12–14 e V2-A | **Cerqueira, Torgo e Mozetič (2020), Evaluating time series forecasting models** ([preprint](https://arxiv.org/abs/1905.11744)). **Cawley e Talbot (2010), On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation** ([artigo](https://jmlr.org/papers/v11/cawley10a.html)). Avaliação cronológica e interpretação dos efeitos de seleção. |

As regras de elegibilidade, os horários de corte e a configuração dos candidatos pertencem a este estudo. As referências fundamentam o procedimento e a leitura, e os dados das execuções dão os resultados.
